# Generador de datos de Chulupi Lang

Este notebook fabrica **pares de oraciones** para pre-entrenar un traductor español–aymara.

Cada par tiene tres versiones de la misma idea:

| Versión | Qué es | Ejemplo |
|---|---|---|
| **Modo B** | Oración ficticia con sufijos y palabras gramaticales **reales** del aymara | `Nayaxa tisqthwa.` |
| **Modo A** | La misma oración con sufijos y palabras gramaticales **inventados** | `Muqixu tisqphyi.` |
| **Español** | La traducción | "Yo voy." |

Las raíces (la parte de la palabra que lleva el significado, como `tisqa` "ir") son inventadas en los dos modos.

**La regla de oro.** La especificación `chulupi_lang_spec.md` (versión 1.1) manda. Cada regla tiene un código `CH-xxx`. El código de este notebook cita esos códigos en comentarios, para que puedas buscar la regla en la especificación.

### Palabras que vas a ver

| Término | Qué significa |
|---|---|
| **raíz** | Parte de la palabra con el significado: `tisqa` "ir", `chimula` "casa". |
| **sufijo** | Trozo que se pega al final de la raíz y añade información: `-naka` = plural. |
| **morfema** | Cualquier pieza con significado propio: una raíz o un sufijo. |
| **id de morfema** | Nombre corto de un sufijo en la especificación: `PL` (plural), `P1` (1.ª persona), `AFF` (afirmativo). La sección 3 tiene la tabla completa. |
| **grafema** | Una "letra" del alfabeto, aunque se escriba con varios caracteres: `ch'` es un solo grafema. |
| **glosa** | Explicación pieza a pieza de una palabra: `tisq-th-wa` = ir-1>3.SIMP-AFF. |
| **idea** | Descripción en datos (JSON) de lo que dice la oración: verbo, sujeto, tiempo… |
| **par** | Una idea con sus tres versiones: B, A y español. |

### Cómo está organizado

| Sección | Qué hace |
|---|---|
| 0 | Introducción y parámetros |
| 1 | Lee la configuración de la especificación |
| 2 | Alfabeto y reglas de sonidos (fonotáctica) |
| 3 | Motor que construye palabras |
| 4 | Pruebas de referencia del motor |
| 5 | Léxico: lee las palabras del diccionario y revisa su formato |
| 6 | Raíces inventadas: las fabrica y las filtra contra palabras reales del aymara |
| 7–13 | Español, oraciones, ideas, generación, verificación y uso *(próximas fases)* |

**Para usarlo:** cambia los valores de la celda "Parámetros" y luego usa *Entorno de ejecución → Ejecutar todo* (Colab) o *Run All* (VS Code).

## 0. Introducción y parámetros

**Qué hace esta sección.**
1. Detecta si el notebook corre en Google Colab o en tu computadora.
2. Instala lo que falte. Solo hace falta PyYAML, que lee los bloques de configuración de la especificación.
3. Reúne en una celda todo lo que puedes cambiar (los **parámetros**).
4. Busca los archivos de datos.

**Reglas que implementa.** Ninguna regla lingüística. Prepara el entorno.

**Qué entra.** Los archivos de la carpeta `datos/`: la especificación, `regresion_ch710.json` y `casos_prueba.json` son obligatorios. `lexico_sin_raices.csv` y `palabras_reales.txt` son opcionales.

**Qué sale.** Las variables `CARPETA_DATOS` y `CARPETA_SALIDA`.

**Ejemplo.** Al final, la carpeta de salida tendrá pares como este:
- B: `Nayaxa tisqthwa.`
- A: `Muqixu tisqphyi.`
- Español: "Yo voy."

In [1]:
# 0.1 · Entorno: ¿Colab o local? ¿Falta algún paquete?
import sys
import importlib.util

EN_COLAB = "google.colab" in sys.modules   # Colab carga este módulo al arrancar

# Colab ya trae pandas, matplotlib y PyYAML. En una computadora pueden faltar.
# PyYAML lee los bloques ```yaml de la especificación (sección 1).
PAQUETES = {"yaml": "pyyaml", "pandas": "pandas", "matplotlib": "matplotlib"}
faltan = [pip_nombre for modulo, pip_nombre in PAQUETES.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    %pip install -q {" ".join(faltan)}

print("Entorno:", "Google Colab" if EN_COLAB else "local (VS Code o Jupyter)")
print("Python", sys.version.split()[0])

Entorno: local (VS Code o Jupyter)
Python 3.11.3


### Parámetros

Esta es la **única celda que necesitas cambiar**.

| Parámetro | Qué controla | Valores |
|---|---|---|
| `N_PARES` | Cuántos pares se generan | Número entero. 10.000 tarda pocos minutos. |
| `SEMILLA` | Semilla del azar. La misma semilla da siempre el mismo archivo. | Número entero |
| `MODO_FILTRO` | Cómo se filtran las raíces que se parecen a palabras reales (CH-306) | `"guia"` o `"corpus"` |
| `CARPETA_SALIDA` | Dónde se guardan los resultados | Nombre de carpeta |
| `PROCESOS` | Cuántos procesos trabajan a la vez al generar | 1 o más |
| `FUENTE_COLAB` | Solo en Colab: de dónde salen los datos | `"drive"` o `"subir"` |
| `CARPETA_DRIVE` | Solo en Colab con `"drive"`: tu carpeta del proyecto en Google Drive | Ruta |

**Una semilla** es un número que fija el azar. Con la misma semilla, el programa "sortea" lo mismo cada vez.

In [2]:
# 0.2 · Parámetros: cambia aquí lo que necesites.
N_PARES = 10_000          # número de pares que se generan
SEMILLA = 20261120        # semilla de la generación de pares (las raíces usan otra, fija: CH-309)
MODO_FILTRO = "guia"      # "guia" o "corpus" (CH-306)
CARPETA_SALIDA = "salida" # carpeta de resultados (en Colab con Drive, dentro de CARPETA_DRIVE)
PROCESOS = 2              # procesos en paralelo para generar (Colab gratuito tiene 2 núcleos)

# Solo para Google Colab
FUENTE_COLAB = "drive"                        # "drive": lee de Google Drive; "subir": eliges los archivos a mano
CARPETA_DRIVE = "/content/drive/MyDrive/chulupi"  # debe contener la subcarpeta datos/

# Comprobaciones: un valor mal escrito se detecta aquí y no a mitad de la generación.
if not (isinstance(N_PARES, int) and N_PARES > 0):
    raise ValueError("N_PARES debe ser un número entero mayor que 0.")
if not isinstance(SEMILLA, int):
    raise ValueError("SEMILLA debe ser un número entero.")
if MODO_FILTRO not in ("guia", "corpus"):
    raise ValueError('MODO_FILTRO debe ser "guia" o "corpus" (CH-306).')
if not (isinstance(PROCESOS, int) and PROCESOS >= 1):
    raise ValueError("PROCESOS debe ser un número entero mayor o igual que 1.")
if FUENTE_COLAB not in ("drive", "subir"):
    raise ValueError('FUENTE_COLAB debe ser "drive" o "subir".')
print("Parámetros correctos.")

Parámetros correctos.


### Archivos de datos

- **En Colab con `"drive"`:** se monta Google Drive. Colab te pedirá permiso la primera vez. Los datos se leen de `CARPETA_DRIVE/datos/` y los resultados se guardan en `CARPETA_DRIVE/salida/`, así no se pierden al cerrar Colab.
- **En Colab con `"subir"`:** aparece un botón para elegir los archivos de tu computadora. Los resultados quedan en Colab y **se borran al cerrar la sesión**: descárgalos al terminar.
- **En local:** se usa la carpeta `datos/` que está junto al notebook.

In [3]:
# 0.3 · Ubicar la carpeta de datos y la de salida.
from pathlib import Path

NOMBRES_SPEC = ["chulupi_lang_spec.md", "artificial_lang_spec.md"]   # ver PREGUNTAS.md, P-01
OBLIGATORIOS = ["regresion_ch710.json", "casos_prueba.json"]
OPCIONALES = ["lexico_sin_raices.csv", "palabras_reales.txt"]


def tiene_spec(carpeta):
    """Dice si una carpeta contiene la especificación.

    Parámetros:
        carpeta: ruta (Path) que se revisa.
    Devuelve:
        True si dentro está alguno de los nombres de NOMBRES_SPEC.
    Reglas: ninguna (preparación).
    Ejemplo:
        >>> tiene_spec(Path("datos"))
        True
    """
    return any((carpeta / n).exists() for n in NOMBRES_SPEC)


if EN_COLAB and FUENTE_COLAB == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DATOS = Path(CARPETA_DRIVE) / "datos"
    BASE_SALIDA = Path(CARPETA_DRIVE)
elif EN_COLAB:
    from google.colab import files
    CARPETA_DATOS = Path.cwd() / "datos"
    CARPETA_DATOS.mkdir(exist_ok=True)
    print("Elige los archivos de tu carpeta datos/: la especificación,", ", ".join(OBLIGATORIOS),
          "y, si los tienes,", ", ".join(OPCIONALES) + ".")
    for nombre, contenido in files.upload().items():
        (CARPETA_DATOS / nombre).write_bytes(contenido)
    BASE_SALIDA = Path.cwd()
else:
    # En local, el notebook suele abrirse desde la carpeta del proyecto; probamos también la carpeta de arriba.
    candidatas = [Path.cwd() / "datos", Path.cwd(), Path.cwd().parent / "datos"]
    CARPETA_DATOS = next((c for c in candidatas if tiene_spec(c)), candidatas[0])
    BASE_SALIDA = CARPETA_DATOS.parent

RUTA_SPEC = next((CARPETA_DATOS / n for n in NOMBRES_SPEC if (CARPETA_DATOS / n).exists()), None)
faltan = ([] if RUTA_SPEC else ["la especificación (" + " o ".join(NOMBRES_SPEC) + ")"]) + \
         [n for n in OBLIGATORIOS if not (CARPETA_DATOS / n).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan archivos en {CARPETA_DATOS}: {', '.join(faltan)}. "
                            "Revisa la sección 'Archivos de datos' de arriba.")

CARPETA_SALIDA = Path(CARPETA_SALIDA) if Path(CARPETA_SALIDA).is_absolute() else BASE_SALIDA / CARPETA_SALIDA
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

print("Datos:        ", CARPETA_DATOS)
print("Especificación:", RUTA_SPEC.name)
print("Salida:       ", CARPETA_SALIDA)
for n in OPCIONALES:
    print(f"{n}: {'sí está' if (CARPETA_DATOS / n).exists() else 'no está (se usará la alternativa de la especificación)'}")

Datos:         /Users/mazenabuhamdan/Desktop/Chulupi-lang/datos
Especificación: artificial_lang_spec.md
Salida:        /Users/mazenabuhamdan/Desktop/Chulupi-lang/salida
lexico_sin_raices.csv: no está (se usará la alternativa de la especificación)
palabras_reales.txt: sí está


## 1. Configuración

**Qué hace esta sección.** La especificación trae su configuración en bloques de texto marcados como ` ```yaml `. YAML es un formato de texto para datos (listas y tablas). Esta sección:
1. busca todos esos bloques en la especificación;
2. los lee y los guarda en el diccionario `CONFIG`;
3. comprueba que estén todos.

Todo el código que sigue lee sus tablas de `CONFIG`, no de valores escritos a mano. Así, si el equipo cambia la especificación (por ejemplo, el orden de las palabras en la v1.2), basta con volver a ejecutar el notebook.

**Reglas que implementa.** Carga los datos de CH-101 (alfabeto), §2 (fonotáctica), §3 (generador de raíces), CH-504 y CH-505 (morfemas), §6 (slots), §8 (paradigmas), CH-901 a CH-917 (sintaxis), CH-1102 (esquema de la idea) y CH-1104 a CH-1113 (pesos).

**Qué entra.** El texto de la especificación.

**Qué sale.** `CONFIG["alfabeto"]`, `CONFIG["morfemas"]`, `CONFIG["sintaxis"]`, etc. También `ORDEN_CONSTITUYENTES`, el orden de las partes de la oración (CH-901).

**Ejemplo.** De `CONFIG["morfemas"]` sale que el sufijo de 1.ª persona `P1` es `-tha` en B y `-phi` en A. Con eso, el motor construye:
- B: `tisqthwa` · A: `tisqphyi` · Español: "voy".

In [4]:
# 1.1 · Extraer y leer los bloques yaml de la especificación.
import re
import json
import yaml
import pandas as pd
from IPython.display import display, Markdown

TEXTO_SPEC = RUTA_SPEC.read_text(encoding="utf-8")
VERSION_SPEC = re.search(r"\*\*Versión:\*\*\s*([\d.]+)", TEXTO_SPEC).group(1)

# Bloques que el generador necesita, con la sección de la especificación de donde salen.
BLOQUES_REQUERIDOS = {
    "alfabeto": "§1", "fonotactica": "§2", "generador_raices": "§3", "morfemas": "§5",
    "slots": "§6", "paradigmas": "§8", "sintaxis": "§9", "idea.schema": "§11", "pesos": "§11",
}

# Un valor del tipo "[NP, NP] | null" quiere decir "una lista de dos NP, o nada".
RE_ALTERNATIVA = re.compile(r'^(\s*[\w.]+:\s+)([\[{"].*\|.*?)(\s+#.*)?$')


def extraer_bloques_yaml(texto):
    """Encuentra los bloques ```yaml de la especificación.

    El nombre de cada bloque sale de su primera línea, un comentario como
    "# alfabeto.yaml" o "# sintaxis (parte de slots.yaml)".

    Parámetros:
        texto: la especificación completa.
    Devuelve:
        Diccionario {nombre del bloque: texto yaml}. El nombre va sin ".yaml".
    Reglas: todas las que tienen tabla de configuración (ver la celda de arriba).
    Ejemplo:
        >>> list(extraer_bloques_yaml(TEXTO_SPEC))[:3]
        ['alfabeto', 'fonotactica', 'generador_raices']
    """
    bloques = {}
    for m in re.finditer(r"```yaml\n(.*?)```", texto, re.S):
        fuente = m.group(1)
        nombre = re.match(r"#\s*([\w.]+)", fuente.lstrip().splitlines()[0])
        if not nombre:
            raise ValueError("Un bloque yaml de la especificación no empieza con '# nombre'.")
        clave = nombre.group(1).removesuffix(".yaml")
        if clave in bloques:
            raise ValueError(f"El bloque yaml '{clave}' aparece dos veces en la especificación.")
        bloques[clave] = fuente
    return bloques


def cargar_yaml(fuente):
    """Lee un bloque yaml. Si no es yaml válido, entrecomilla las alternativas "X | Y".

    Parámetros:
        fuente: texto del bloque.
    Devuelve:
        (datos, corregido). corregido = True si hubo que entrecomillar líneas.
    Reglas: CH-1102 (su bloque usa "X | Y", que no es yaml válido; ver PREGUNTAS.md, P-02).
    Ejemplo:
        >>> cargar_yaml("od_coord: [NP, NP] | null")
        ({'od_coord': '[NP, NP] | null'}, True)
    """
    try:
        return yaml.safe_load(fuente), False
    except yaml.YAMLError:
        lineas = []
        for linea in fuente.splitlines():
            m = RE_ALTERNATIVA.match(linea)
            lineas.append(m.group(1) + json.dumps(m.group(2), ensure_ascii=False) + (m.group(3) or "") if m else linea)
        return yaml.safe_load("\n".join(lineas)), True


BLOQUES = extraer_bloques_yaml(TEXTO_SPEC)
faltan = [b for b in BLOQUES_REQUERIDOS if b not in BLOQUES]
if faltan:
    raise ValueError(f"La especificación no tiene los bloques yaml: {faltan}")

CONFIG, CORREGIDOS = {}, []
for nombre, fuente in BLOQUES.items():
    CONFIG[nombre], corregido = cargar_yaml(fuente)
    if corregido:
        CORREGIDOS.append(nombre)

# El parámetro MODO_FILTRO manda sobre el valor escrito en la especificación (CH-306).
modo_spec = CONFIG["generador_raices"]["filtro_real"]["modo"]
CONFIG["generador_raices"]["filtro_real"]["modo"] = MODO_FILTRO

# CH-901: el orden de las partes de la oración se lee de la configuración.
ORDEN_CONSTITUYENTES = CONFIG["sintaxis"]["orden_constituyentes"]

if VERSION_SPEC != "1.1":
    print(f"AVISO: la especificación es la versión {VERSION_SPEC}; este notebook implementa la 1.1.")
print(f"Especificación versión {VERSION_SPEC}: {len(BLOQUES)} bloques yaml leídos.")
if CORREGIDOS:
    print("Bloques leídos con corrección (ver PREGUNTAS.md, P-02):", ", ".join(CORREGIDOS))
print(f"Modo del filtro de raíces: {MODO_FILTRO} (la especificación dice '{modo_spec}').")
print("Orden de constituyentes (CH-901):", " – ".join(ORDEN_CONSTITUYENTES))
display(pd.DataFrame(
    [{"bloque": n, "sección": BLOQUES_REQUERIDOS.get(n, "§7 (CH-710)"),
      "contenido": ", ".join(list(CONFIG[n])[:5]) + (" …" if len(CONFIG[n]) > 5 else ""),
      "corregido": "sí" if n in CORREGIDOS else ""} for n in BLOQUES]))

Especificación versión 1.1: 10 bloques yaml leídos.
Bloques leídos con corrección (ver PREGUNTAS.md, P-02): idea.schema
Modo del filtro de raíces: guia (la especificación dice 'guia').
Orden de constituyentes (CH-901): S – JANI – T – L – COMCAUSE – OI – OD – V


,bloque,sección,contenido,corregido
0,alfabeto,§1,"version, normalizacion, vocales, vocales_larga...",
1,fonotactica,§2,"clases, raiz, palabra",
2,generador_raices,§3,"semilla_raices, silabas, plantillas, inicio, i...",
3,morfemas,§5,"sufijos, palabras_gramaticales",
4,slots,§6,"nombre, verbo",
5,regresion_raices_reales,§7 (CH-710),casos,
6,paradigmas,§8,"verbo_prueba, celdas, oracional, predicados_no...",
7,sintaxis,§9,"orden_constituyentes, posicion_wa, top, q_yn_e...",
8,idea.schema,§11,"predicado, modalidad, tiempo, aspecto, verbo …",sí
9,pesos,§11,"predicado_modalidad, tiempo, aspecto, persona_...",


## 2. Alfabeto y fonotáctica

**Qué hace esta sección.** Define las herramientas para revisar la escritura:
- `normalizar`: unifica la escritura (apóstrofo, vocales largas, minúsculas).
- `segmentar`: corta una palabra en grafemas.
- `clases`: cambia cada grafema por su clase: `C` consonante, `V` vocal corta, `L` vocal larga.
- `validar_palabra`: revisa cualquier palabra de salida.
- `validar_raiz`: revisa una raíz inventada, con reglas más estrictas.

**Términos.**
- **Fonotáctica**: las reglas sobre qué sonidos pueden ir juntos y en qué orden.
- **Sílaba**: grupo de sonidos con una vocal en el centro: `tis.qa` tiene dos.
- **Coda**: consonante que cierra una sílaba. En `tis.qa`, la `s` es coda.
- **Vocal larga**: `ä ï ü`. Se pronuncia más larga que `a i u`.
- **Consonante aspirada**: se pronuncia con un soplo de aire. Se escribe con `h`: `ph th chh kh qh`.
- **Consonante glotalizada**: se pronuncia con un pequeño golpe de garganta. Se escribe con apóstrofo: `p' t' ch' k' q'`.
- **Laringalizada**: aspirada o glotalizada.
- **NFC**: una forma estándar de guardar letras con tilde o diéresis en el ordenador. Así `ä` es siempre un solo carácter.

**Reglas que implementa.** CH-101 a CH-106 (alfabeto) y CH-201 a CH-209 (fonotáctica).

**Qué entra.** Palabras o raíces. **Qué sale.** Listas de grafemas y listas de errores (vacías si todo está bien).

**Ejemplo.** `segmentar("ch'ixrthwa")` → `ch'` `i` `x` `r` `th` `w` `a`. Las clases son `CVCCCCV`: la palabra es válida.
- B: `ch'ixrthwa` · A: `ch'ixrphyi` · Español: "veo".

In [5]:
# 2.1 · Alfabeto (§1) y fonotáctica (§2), leídos de CONFIG.
import unicodedata

ALF = CONFIG["alfabeto"]
FON = CONFIG["fonotactica"]

GRAFEMAS = ALF["orden_segmentacion"]                     # CH-105: el más largo primero
VOCALES_CORTAS = set(ALF["vocales"])
LARGA = dict(ALF["vocales_largas"])                      # a → ä, i → ï, u → ü
VOCALES_LARGAS = set(LARGA.values())
CONSONANTES = {g for lista in ALF["consonantes"].values() for g in lista}
APOSTROFO = ALF["normalizacion"]["apostrofo_interno"]   # CH-102: U+0027
LARINGALIZADOS = set(FON["raiz"]["laringalizados"])
PUNTO_DE_ARTICULACION = ALF["punto_de_articulacion"]
RE_RAIZ = re.compile(FON["raiz"]["regex_clases"])        # CH-201
RE_PALABRA = re.compile(FON["palabra"]["regex_clases"])  # CH-202, CH-203, CH-204, CH-208

# Comprobaciones de la propia configuración.
assert set(GRAFEMAS) == VOCALES_CORTAS | VOCALES_LARGAS | CONSONANTES, "alfabeto.yaml: el orden de segmentación no cubre el alfabeto"
assert not set(ALF["prohibidos"]) & set(GRAFEMAS), "alfabeto.yaml: una letra prohibida está en el alfabeto"
assert APOSTROFO == "\u0027", "CH-102: el apóstrofo interno debe ser U+0027"


def normalizar(texto):
    """Unifica la escritura antes de comparar o validar.

    Pasa el texto a NFC, cambia los apóstrofos tipográficos por U+0027 y pasa a minúsculas.

    Parámetros:
        texto: cualquier cadena.
    Devuelve:
        La cadena normalizada.
    Reglas: CH-102 (apóstrofo), CH-103 (vocal larga en NFC), CH-106 (los validadores trabajan en minúsculas).
    Ejemplo:
        >>> normalizar("Lamch’thwa")
        "lamch'thwa"
    """
    t = unicodedata.normalize(ALF["normalizacion"]["unicode"], texto)   # CH-103: a + U+0308 → ä
    for ap in ALF["normalizacion"]["apostrofos_a_convertir"]:           # CH-102
        t = t.replace(ap, APOSTROFO)
    return t.lower()


def segmentar(palabra):
    """Corta una palabra en grafemas, tomando siempre el más largo posible.

    Parámetros:
        palabra: cadena ya normalizada (minúsculas, NFC).
    Devuelve:
        Lista de grafemas.
    Lanza:
        ValueError si un carácter no forma ningún grafema (CH-104: letras prohibidas).
    Reglas: CH-101, CH-104, CH-105.
    Ejemplo:
        >>> segmentar("ch'ixrthwa")
        ["ch'", 'i', 'x', 'r', 'th', 'w', 'a']
    """
    out, i = [], 0
    while i < len(palabra):
        for g in GRAFEMAS:                                   # CH-105: el orden ya pone primero los largos
            if palabra.startswith(g, i):
                out.append(g)
                i += len(g)
                break
        else:
            raise ValueError(f"CH-104: carácter no permitido en {palabra!r} (posición {i}: {palabra[i]!r})")
    return out


def clases(grafemas):
    """Cambia cada grafema por su clase.

    Parámetros:
        grafemas: lista que devuelve segmentar().
    Devuelve:
        Cadena con C (consonante), V (vocal corta) y L (vocal larga).
    Reglas: notación de §2.
    Ejemplo:
        >>> clases(segmentar("tisqä"))
        'CVCCL'
    """
    return "".join("V" if g in VOCALES_CORTAS else "L" if g in VOCALES_LARGAS else "C" for g in grafemas)


def validar_palabra(palabra):
    """Revisa una palabra de salida (modo B o A).

    Parámetros:
        palabra: palabra en minúsculas y sin puntuación.
    Devuelve:
        Lista de errores, cada uno con su regla. Vacía si la palabra es válida.
    Reglas: CH-101 a CH-105, CH-202 a CH-208 (CH-709: toda palabra construida pasa por aquí).
    Ejemplo:
        >>> validar_palabra("tisqthwa")
        []
        >>> validar_palabra("tisqaiwa")
        ['CH-202/203/204/208: estructura inválida CVCCVVCV']
    """
    try:
        g = segmentar(palabra)
    except ValueError as e:
        return [str(e)]
    errores = []
    if not RE_PALABRA.match(clases(g)):
        errores.append(f"CH-202/203/204/208: estructura inválida {clases(g)}")
    if g[0] in FON["palabra"]["iniciales_prohibidas"]:
        errores.append(f"CH-202: empieza por {g[0]}")
    for cadena in FON["palabra"]["cadenas_prohibidas"]:
        if cadena in palabra:
            errores.append(f"CH-207: contiene {cadena}")
    return errores


def es_coda(grafemas, i):
    """Dice si el grafema i cierra sílaba: es consonante y le sigue otra consonante.

    Parámetros:
        grafemas: lista de grafemas de una raíz.
        i: posición que se revisa.
    Devuelve:
        True si es coda.
    Reglas: CH-201 (si hay dos consonantes entre vocales, la primera es coda).
    Ejemplo:
        >>> es_coda(segmentar("tisqa"), 2)   # la s de tis.qa
        True
    """
    return (grafemas[i] in CONSONANTES and i + 1 < len(grafemas) and grafemas[i + 1] in CONSONANTES)


def validar_raiz(raiz):
    """Revisa una raíz inventada con las reglas estrictas de la raíz.

    Parámetros:
        raiz: la raíz, en minúsculas.
    Devuelve:
        Lista de errores con su regla. Vacía si la raíz es válida.
    Reglas: CH-201, CH-202, CH-205, CH-206, CH-207, CH-209.
    Ejemplo:
        >>> validar_raiz("q'aspi")
        []
        >>> validar_raiz("tissa")
        ['CH-206: coda igual al inicio siguiente (s + s)']
    """
    try:
        g = segmentar(raiz)
    except ValueError as e:
        return [str(e)]
    R = FON["raiz"]
    errores = []
    if not RE_RAIZ.match(clases(g)):
        errores.append(f"CH-201: plantilla inválida {clases(g)}")
    if g[0] in FON["palabra"]["iniciales_prohibidas"]:
        errores.append(f"CH-202: empieza por {g[0]}")
    for v in g:
        if (v in VOCALES_CORTAS or v in VOCALES_LARGAS) and v not in R["vocales_permitidas"]:
            errores.append(f"CH-209: vocal {v} no permitida en la raíz")
    for i, c in enumerate(g):
        if not es_coda(g, i):
            continue
        sig = g[i + 1]
        if c in LARINGALIZADOS:
            errores.append(f"CH-205: laringalizada {c} en coda")
        elif c not in R["codas_permitidas"]:
            errores.append(f"CH-206: {c} no puede ser coda")
        if R["prohibido_coda_igual_a_inicio"] and c == sig:
            errores.append(f"CH-206: coda igual al inicio siguiente ({c} + {sig})")
        if c == "l" and sig in R["prohibido_l_antes_de"]:
            errores.append(f"CH-207: l + {sig}")
    lar = {x for x in g if x in LARINGALIZADOS}
    if len(lar) > R["max_laringalizados_distintos"]:
        errores.append(f"CH-205: más de un laringalizado distinto {sorted(lar)}")
    if R["armonia_mismo_punto"]:
        for punto, serie in PUNTO_DE_ARTICULACION.items():
            presentes = {x for x in g if x in serie}
            if len(presentes) > 1:
                errores.append(f"CH-205: oclusivas del punto {punto} distintas {sorted(presentes)}")
    for cadena in FON["palabra"]["cadenas_prohibidas"]:
        if cadena in raiz:
            errores.append(f"CH-207: contiene {cadena}")
    return errores


print("Grafemas:", len(GRAFEMAS), "·", " ".join(GRAFEMAS))
for w in ["tisqthwa", "ch'ixrthwa", "tisqaiwa", "tishwa"]:
    print(f"{w:12} errores: {validar_palabra(w) or 'ninguno'}")

Grafemas: 32 · chh ch' ch ph p' th t' kh k' qh q' ll a i u ä ï ü p t k q s j x m n ñ l r w y
tisqthwa     errores: ninguno
ch'ixrthwa   errores: ninguno
tisqaiwa     errores: ['CH-202/203/204/208: estructura inválida CVCCVVCV']
tishwa       errores: ["CH-104: carácter no permitido en 'tishwa' (posición 3: 'h')"]


### Pruebas de la sección 2

Una **prueba** compara lo que produce el código con lo que dice la especificación. Hay al menos una por regla, con los ejemplos de la propia especificación.

Si una prueba falla, el notebook **se detiene** con un mensaje que dice qué falló. Así nadie genera datos con un error.

Usamos `unittest`, que viene con Python. Funciona igual en Colab y en local.

In [6]:
# 2.2 · Herramienta para correr pruebas y detener el notebook si alguna falla.
import unittest


def correr_pruebas(*clases_prueba, titulo):
    """Corre clases de prueba de unittest y detiene el notebook si alguna falla.

    Parámetros:
        clases_prueba: una o varias clases unittest.TestCase.
        titulo: nombre corto que aparece en el mensaje.
    Devuelve:
        Nada. Imprime el resultado de cada prueba.
    Lanza:
        RuntimeError si alguna prueba falla. Eso detiene "Ejecutar todo".
    Reglas: ninguna (infraestructura de pruebas).
    Ejemplo:
        >>> correr_pruebas(PruebasAlfabeto, titulo="alfabeto")
    """
    cargador = unittest.defaultTestLoader
    suite = unittest.TestSuite(cargador.loadTestsFromTestCase(c) for c in clases_prueba)
    res = unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(suite)
    if not res.wasSuccessful():
        raise RuntimeError(f"Fallaron {len(res.failures) + len(res.errors)} pruebas de {titulo}. "
                           "Lee el detalle de arriba. El notebook se detiene aquí.")
    print(f"✔ {titulo}: {res.testsRun} pruebas en verde.")


def tiene_regla(errores, regla):
    """Dice si alguna línea de una lista de errores cita una regla.

    Parámetros:
        errores: lista de cadenas (de validar_palabra o validar_raiz).
        regla: código, por ejemplo "CH-206".
    Devuelve:
        True si alguna línea contiene ese código.
    Reglas: ninguna (ayuda para las pruebas).
    Ejemplo:
        >>> tiene_regla(validar_raiz("tissa"), "CH-206")
        True
    """
    return any(regla in e for e in errores)

In [7]:
# 2.3 · Pruebas de §1 (alfabeto) y §2 (fonotáctica). Ejemplos tomados de la especificación.
class PruebasAlfabeto(unittest.TestCase):
    def test_CH101_letras_permitidas(self):
        """CH-101: tisqthwa (B) y tisqphyi (A) usan solo letras del alfabeto."""
        self.assertEqual(segmentar("tisqthwa"), ["t", "i", "s", "q", "th", "w", "a"])
        self.assertEqual(segmentar("tisqphyi"), ["t", "i", "s", "q", "ph", "y", "i"])

    def test_CH102_apostrofo(self):
        """CH-102: los apóstrofos tipográficos pasan a U+0027."""
        self.assertEqual(normalizar("lamch’thwa"), "lamch'thwa")
        self.assertEqual(normalizar("lamch’phyi"), "lamch'phyi")
        for ap in ["’", "‘", "ʼ", "´", "`"]:
            self.assertEqual(normalizar(f"q{ap}aspi"), "q'aspi")

    def test_CH103_vocal_larga_nfc(self):
        """CH-103: a + diéresis combinante (U+0308) pasa a ä precompuesta."""
        self.assertEqual(normalizar("tisqa\u0308"), "tisqä")
        self.assertEqual(segmentar(normalizar("tisqa\u0308")), ["t", "i", "s", "q", "ä"])

    def test_CH104_letras_prohibidas(self):
        """CH-104: tisqe y tishwa son inválidas; tisqhwa es válida."""
        self.assertTrue(tiene_regla(validar_palabra("tisqe"), "CH-104"))
        self.assertTrue(tiene_regla(validar_palabra("tishwa"), "CH-104"))
        self.assertEqual(validar_palabra("tisqhwa"), [])
        for w in ["tisqo", "bata", "tisq-wa", "tisq1a", "tisqá"]:
            self.assertTrue(tiene_regla(validar_palabra(w), "CH-104"), w)

    def test_CH105_segmentacion(self):
        """CH-105: el grafema más largo primero (ch' antes que ch, chh antes que ch)."""
        self.assertEqual(segmentar("ch'ixrthwa"), ["ch'", "i", "x", "r", "th", "w", "a"])
        self.assertEqual(segmentar("ch'ixrphyi"), ["ch'", "i", "x", "r", "ph", "y", "i"])
        self.assertEqual(segmentar("chhalla"), ["chh", "a", "ll", "a"])

    def test_CH106_minusculas(self):
        """CH-106: los validadores trabajan sobre minúsculas (Ä pasa a ä)."""
        self.assertEqual(normalizar("Äta"), "äta")


class PruebasFonotactica(unittest.TestCase):
    def test_CH201_plantilla_raiz(self):
        """CH-201: lamch'a válida; lamch'tra inválida (4 consonantes seguidas)."""
        self.assertEqual(validar_raiz("lamch'a"), [])
        self.assertTrue(tiene_regla(validar_raiz("lamch'tra"), "CH-201"))
        self.assertTrue(tiene_regla(validar_raiz("tisq"), "CH-201"))      # la última sílaba es CV
        self.assertTrue(tiene_regla(validar_raiz("pa"), "CH-201"))        # mínimo 2 sílabas

    def test_CH202_inicio(self):
        """CH-202: xiku, rupa y smiqa son inválidas."""
        for w in ["xiku", "rupa", "smiqa"]:
            self.assertNotEqual(validar_palabra(w), [], w)
            self.assertNotEqual(validar_raiz(w), [], w)
        self.assertTrue(tiene_regla(validar_palabra("xiku"), "CH-202"))

    def test_CH203_final(self):
        """CH-203: tisqä válida; tisqth (B) y tisqph (A) inválidas."""
        self.assertEqual(validar_palabra("tisqä"), [])
        self.assertNotEqual(validar_palabra("tisqth"), [])
        self.assertNotEqual(validar_palabra("tisqph"), [])

    def test_CH204_sin_vocales_seguidas(self):
        """CH-204: tisqaiwa (B) y tisqauyi (A) inválidas."""
        self.assertNotEqual(validar_palabra("tisqaiwa"), [])
        self.assertNotEqual(validar_palabra("tisqauyi"), [])

    def test_CH205_laringalizadas(self):
        """CH-205: q'aspi y t'ant'a válidas; q'aspha y t'ata inválidas."""
        self.assertEqual(validar_raiz("q'aspi"), [])
        self.assertEqual(validar_raiz("t'ant'a"), [])
        self.assertTrue(tiene_regla(validar_raiz("q'aspha"), "CH-205"))
        self.assertTrue(tiene_regla(validar_raiz("t'ata"), "CH-205"))
        self.assertTrue(tiene_regla(validar_raiz("lap'ta"), "CH-205"))   # laringalizada en coda

    def test_CH206_codas(self):
        """CH-206: pajlla válida; tañqa (coda ñ) y tissa (s + s) inválidas."""
        self.assertEqual(validar_raiz("pajlla"), [])
        self.assertTrue(tiene_regla(validar_raiz("tañqa"), "CH-206"))
        self.assertTrue(tiene_regla(validar_raiz("tissa"), "CH-206"))

    def test_CH207_l_mas_l(self):
        """CH-207: kalla se lee ka.lla; ninguna palabra contiene lll."""
        self.assertEqual(segmentar("kalla"), ["k", "a", "ll", "a"])
        self.assertTrue(tiene_regla(validar_palabra("kallla"), "CH-207"))
        self.assertNotEqual(validar_raiz("kallla"), [])

    def test_CH208_grupos(self):
        """CH-208: hasta 6 consonantes seguidas fuera de la raíz (jiskht'kthti); 7 no."""
        self.assertEqual(validar_palabra("tisqkiti"), [])
        self.assertEqual(validar_palabra("tisqñuji"), [])
        self.assertEqual(validar_palabra("jiskht'kthti"), [])
        self.assertNotEqual(validar_palabra("a" + "k" * 7 + "a"), [])

    def test_CH209_raiz_sin_vocal_larga(self):
        """CH-209: tisqa válida; tïsqa inválida."""
        self.assertEqual(validar_raiz("tisqa"), [])
        self.assertTrue(tiene_regla(validar_raiz("tïsqa"), "CH-209"))


correr_pruebas(PruebasAlfabeto, PruebasFonotactica, titulo="alfabeto y fonotáctica (§1, §2)")

test_CH101_letras_permitidas (__main__.PruebasAlfabeto.test_CH101_letras_permitidas)
CH-101: tisqthwa (B) y tisqphyi (A) usan solo letras del alfabeto. ... 

ok


test_CH102_apostrofo (__main__.PruebasAlfabeto.test_CH102_apostrofo)
CH-102: los apóstrofos tipográficos pasan a U+0027. ... 

ok


test_CH103_vocal_larga_nfc (__main__.PruebasAlfabeto.test_CH103_vocal_larga_nfc)
CH-103: a + diéresis combinante (U+0308) pasa a ä precompuesta. ... 

ok


... (22 outputs left out)

test_CH209_raiz_sin_vocal_larga (__main__.PruebasFonotactica.test_CH209_raiz_sin_vocal_larga)
CH-209: tisqa válida; tïsqa inválida. ... 

ok


----------------------------------------------------------------------
Ran 15 tests in 0.005s

OK


✔ alfabeto y fonotáctica (§1, §2): 15 pruebas en verde.


## 3. Motor morfológico

**Qué hace esta sección.** Construye una palabra a partir de una raíz y una lista de sufijos. Es el corazón del generador.

**Términos.**
- **Elisión**: caída de una vocal. En `tisqa` + `-tha`, la `a` final de la raíz cae: `tisqtha`.
- **Alargamiento**: un sufijo puede convertir la vocal anterior en larga. Se escribe `¨`: `tisqa` + `-¨` → `tisqä`.
- **`antes`**: qué le hace un sufijo a la vocal corta de lo que tiene delante. `V` = la conserva. `C` = la borra. `—` = no decide.
- **`después`**: qué pasa con la vocal final del propio sufijo cuando le sigue un oracional. `C` = la pierde. `V` = la conserva.
- **Oracional**: sufijo que va al final de la palabra y marca el tipo de oración: `-wa` (afirmativa), `-xa` (tema), `-ti` (pregunta o negación), `-sa` (pregunta con "qué/quién/dónde"). No decide nada: decide el sufijo anterior (CH-702).
- **Alomorfo**: otra forma del mismo sufijo según el contexto. El pasado `-yä-` se escribe `-¨-` ante la 3.ª persona (CH-511).
- **Transición**: en el verbo, una sola terminación marca quién hace la acción y a quién: `1>2` = "yo a ti" ("te veo").
- **Slot**: casilla de la palabra. Cada sufijo tiene su casilla, y las casillas van en orden fijo (§6).

**Reglas que implementa.** CH-501 a CH-505 (tabla de morfemas), CH-511 a CH-513, CH-601 a CH-606 (orden de los sufijos) y CH-701 a CH-709 (algoritmo de construcción).

**Qué entra.** Una raíz (o una palabra gramatical en su forma B), una lista de ids de morfemas y el modo (`"B"` o `"A"`).

**Qué sale.** La palabra construida. Con `traza=True`, también cada paso y cada decisión.

**Ejemplo** (`tisqa` "ir" + `P1` + `AFF`):

| Paso | Modo B | Modo A |
|---|---|---|
| raíz | `tisqa` | `tisqa` |
| + P1 (antes C) | borra `a` → `tisqtha` | borra `a` → `tisqphi` |
| + AFF (decide P1, después C) | borra `a` → `tisqthwa` | borra `i` → `tisqphyi` |

Español: "voy".

### Tabla de ids de morfemas

| Id | Qué significa | B | A |
|---|---|---|---|
| `PL` | plural del nombre | `-naka` | `-sulu` |
| `POSS.1` … `POSS.4` | poseedor: mi, tu, su, nuestro (tú y yo) | `-ja -ma -pa -sa` | `-lu -qi -chu -khu` |
| `POSR` | "que tiene" (para "tener") | `-ni` | `-phu` |
| `GEN` | "de" (poseedor) | `-na` | `-mu` |
| `LOC` | "en" | `-na` | `-mu` |
| `ALL` | "a, hacia" (lugar) | `-ru` | `-llu` |
| `COMPL.RU` | "a" (persona que recibe) | `-ru` | `-llu` |
| `ABL` | "de, desde" | `-ta` | `-ku` |
| `COM` | "con" | `-mpi` | `-nku` |
| `BEN` | "para" | `-taki` | `-puli` |
| `UNTIL` | "hasta" | `-kama` | `-chiwi` |
| `CAUSE` | "por, a causa de" | `-layku` | `-tiwqa` |
| `VBLZ.KA` | convierte "en X" en "estar en X" | `-ka` | `-ñu` |
| `VBLZ.LONG` | convierte un nombre o adjetivo en "ser X" | `-¨` | `-¨` |
| `PL.V` | plural del sujeto, en el verbo | `-pxa` | `-mtu` |
| `PROG` | progresivo ("estar + -ndo") | `-ska` | `-nchi` |
| `NEG` | parte de la negación en el verbo | `-ka` | `-ñu` |
| `REM.C` | pasado visto por quien habla | `-yä-` | `-jü-` |
| `REM.L` | pasado no visto | `-tä-` | `-wü-` |
| `P1` `P2` `P3` `P4` | persona del sujeto: yo, tú, él/ella, nosotros (tú y yo) | `-tha -ta -i -tana` | `-phi -ku -u -kilu` |
| `P3.REM` | 3.ª persona en los pasados | `-na` | `-mu` |
| `T12` `T21` `T31` `T32` `T34` | transiciones sujeto>objeto (1>2 = "yo a ti") | `-sma -ista -itu -tama -istu` | `-jmi -ulqa -uqu -kiqu -ulqu` |
| `FUT.1` … `FUT.4` | futuro, por persona | `-¨ -¨ta -ni -ñäni` | `-¨ -¨ku -phu -thäqi` |
| `AFF` | afirmativo | `-wa` | `-yi` |
| `TOP` | tema de la oración (marca el sujeto) | `-xa` | `-xu` |
| `Q.YN` | pregunta de sí o no | `-ti` | `-ji` |
| `NEG.TI` | final de la negación | `-ti` | `-ji` |
| `Q.INF` | pregunta con qué, quién o dónde | `-sa` | `-khu` |

Los tiempos: **SIMP** = presente o pasado simple; **FUT** = futuro; **REM.C** y **REM.L** = pasados remotos. "4.ª persona" = "nosotros" incluyendo a quien escucha.

In [8]:
# 3.1 · Tabla de morfemas (CH-504) y palabras gramaticales (CH-505), leídas de CONFIG.
MORFEMAS = {}
for m in CONFIG["morfemas"]["sufijos"]:
    if m["id"] in MORFEMAS:
        raise ValueError(f"morfemas.yaml: id repetido {m['id']}")
    MORFEMAS[m["id"]] = {**m, "despues": m.get("despues", "-")}   # los oracionales no traen 'despues'

GRAMATICALES = {p["id"]: p for p in CONFIG["morfemas"]["palabras_gramaticales"]}
GRAM_B_A = {p["b"]: p["a"] for p in GRAMATICALES.values()}         # CH-708: forma B → forma A
ORACIONALES = {i for i, m in MORFEMAS.items() if m["antes"] == "-" and m["slot"] in (["N7", "V6"], "N7", "V6")}

print(f"{len(MORFEMAS)} sufijos y {len(GRAMATICALES)} palabras gramaticales.")
print("Oracionales:", ", ".join(sorted(ORACIONALES)))

41 sufijos y 16 palabras gramaticales.
Oracionales: AFF, NEG.TI, Q.INF, Q.YN, TOP


In [9]:
# 3.2 · Algoritmo de construcción de palabras (CH-701 a CH-709).
def construir_palabra(base, ids, modo="B", traza=False):
    """Construye una palabra: base + sufijos, aplicando la caída y el alargamiento de vocales.

    Parámetros:
        base: raíz (de lexico.csv o real) o palabra gramatical en su forma B ("naya", "jani"…).
        ids: lista de ids de morfemas, en orden (CH-601, CH-602). Ejemplo: ["P1", "AFF"].
        modo: "B" (sufijos reales) o "A" (sufijos inventados).
        traza: si es True, devuelve también los pasos y las decisiones.
    Devuelve:
        La palabra. Con traza=True: (palabra, pasos, decisiones).
        'decisiones' es la lista que compara la prueba de paralelismo (CH-710.2):
        borrar, conservar, alargar, proteger, larga-no-borra, protegida-no-borra.
    Lanza:
        ValueError si la palabra final no es válida (CH-709) o el modo no existe.
        KeyError si un id no está en morfemas.yaml.
    Reglas: CH-701 (algoritmo), CH-702 (oracionales), CH-703 (la larga no se borra),
        CH-704 (alargamiento), CH-705 (VBLZ.LONG tras POSR), CH-706 (grupos enteros),
        CH-708 (palabras gramaticales en A), CH-709 (validación), CH-511 (alomorfo de REM.C).
    Ejemplo:
        >>> construir_palabra("tisqa", ["P1", "AFF"], "B")
        'tisqthwa'
        >>> construir_palabra("tisqa", ["P1", "AFF"], "A")
        'tisqphyi'
    """
    if modo not in ("B", "A"):
        raise ValueError("modo debe ser 'B' o 'A'")
    clave = modo.lower()
    forma0 = GRAM_B_A.get(base, base) if modo == "A" else base     # CH-708: la palabra gramatical cambia antes de los sufijos
    g = segmentar(forma0)
    despues_prev, id_prev, protegida = "V", "RAIZ", False
    pasos, decisiones = [f"raíz `{forma0}`"], []
    for k, mid in enumerate(ids):
        if mid not in MORFEMAS:
            raise KeyError(f"El morfema {mid!r} no está en morfemas.yaml")
        m = MORFEMAS[mid]
        f = m[clave]
        alo = m.get("alomorfo")
        if alo and k + 1 < len(ids) and ids[k + 1] == alo["ante"]:   # CH-511: REM.C es -¨- ante P3.REM
            f = alo[clave]
        if id_prev in m.get("invisible_tras", []):                    # CH-705: VBLZ.LONG tras POSR no se escribe
            protegida, despues_prev, id_prev = True, "V", mid
            decisiones.append("proteger")
            pasos.append(f"+ {mid} tras {m['invisible_tras'][0]}: no se escribe; protege `{g[-1]}` (CH-705) → `{''.join(g)}`")
            continue
        if f.startswith("¨"):                                         # CH-704: alarga la vocal anterior
            if g[-1] in VOCALES_CORTAS:
                g[-1] = LARGA[g[-1]]
            if f[1:]:
                g += segmentar(f[1:])
            protegida, despues_prev, id_prev = False, m["despues"], mid
            decisiones.append("alargar")
            pasos.append(f"+ {mid} `{f}`: alarga → `{''.join(g)}`")
            continue
        # CH-702: el oracional (antes "-") no decide; manda la columna 'después' del morfema anterior.
        borrar = m["antes"] == "C" or (m["antes"] == "-" and despues_prev == "C")
        quien = f"antes {m['antes']}" if m["antes"] != "-" else f"antes —; decide {id_prev} (después {despues_prev})"
        ultima = g[-1]
        if borrar and ultima in VOCALES_CORTAS and not protegida:
            g = g[:-1]
            d, texto = "borrar", f"borra `{ultima}`"
        elif borrar and protegida:
            d, texto = "protegida-no-borra", f"`{ultima}` protegida: no se borra (CH-705)"
        elif borrar and ultima in VOCALES_LARGAS:
            d, texto = "larga-no-borra", "no hay vocal corta que borrar (CH-703)"
        else:
            d, texto = "conservar", "conserva la vocal"
        decisiones.append(d)
        protegida = False
        g += segmentar(f)                                             # CH-706: los grupos se escriben enteros
        despues_prev, id_prev = m["despues"], mid
        pasos.append(f"+ {mid} `{f}` ({quien}): {texto} → `{''.join(g)}`")
    palabra = "".join(g)
    errores = validar_palabra(palabra)
    if errores:                                                       # CH-709: error del programa, no se descarta en silencio
        raise ValueError(f"CH-709: palabra inválida {palabra!r} ({base} + {ids}, modo {modo}): {errores}")
    return (palabra, pasos, decisiones) if traza else palabra


def construir_ambos(base, ids):
    """Construye la misma palabra en los dos modos.

    Parámetros:
        base: raíz o palabra gramatical (forma B).
        ids: lista de ids de morfemas.
    Devuelve:
        (forma B, forma A).
    Reglas: CH-501, CH-701.
    Ejemplo:
        >>> construir_ambos("chimula", ["PL", "POSS.1", "LOC"])
        ('chimulanakajana', 'chimulasululumu')
    """
    return construir_palabra(base, ids, "B"), construir_palabra(base, ids, "A")


def mostrar_traza(base, ids):
    """Muestra paso a paso cómo se construye una palabra en los dos modos.

    Parámetros:
        base: raíz o palabra gramatical (forma B).
        ids: lista de ids de morfemas.
    Devuelve:
        Una tabla (DataFrame) con una fila por paso y la salida al final.
    Reglas: CH-701 (formato de los ejemplos de §7.1), CH-710.2 (las decisiones de B y A coinciden).
    Ejemplo:
        >>> mostrar_traza("tisqa", ["P1", "AFF"])
    """
    pb, pasos_b, dec_b = construir_palabra(base, ids, "B", traza=True)
    pa, pasos_a, dec_a = construir_palabra(base, ids, "A", traza=True)
    filas = [{"paso": i, "modo B": b, "modo A": a} for i, (b, a) in enumerate(zip(pasos_b, pasos_a))]
    filas.append({"paso": "salida", "modo B": f"**{pb}**", "modo A": f"**{pa}**"})
    print(f"{base} + {' '.join(ids)} · mismas decisiones en B y A: {'sí' if dec_b == dec_a else 'NO'}")
    return pd.DataFrame(filas).set_index("paso")


with pd.option_context("display.max_colwidth", None):
    display(mostrar_traza("tisqa", ["P1", "AFF"]))
    display(mostrar_traza("chimula", ["POSR", "VBLZ.LONG", "P1", "AFF"]))

tisqa + P1 AFF · mismas decisiones en B y A: sí


,modo B,modo A
paso,,
0,raíz `tisqa`,raíz `tisqa`
1,+ P1 `tha` (antes C): borra `a` → `tisqtha`,+ P1 `phi` (antes C): borra `a` → `tisqphi`
2,+ AFF `wa` (antes —; decide P1 (después C)): borra `a` → `tisqthwa`,+ AFF `yi` (antes —; decide P1 (después C)): borra `i` → `tisqphyi`
salida,**tisqthwa**,**tisqphyi**


chimula + POSR VBLZ.LONG P1 AFF · mismas decisiones en B y A: sí


,modo B,modo A
paso,,
0,raíz `chimula`,raíz `chimula`
1,+ POSR `ni` (antes V): conserva la vocal → `chimulani`,+ POSR `phu` (antes V): conserva la vocal → `chimulaphu`
2,+ VBLZ.LONG tras POSR: no se escribe; protege `i` (CH-705) → `chimulani`,+ VBLZ.LONG tras POSR: no se escribe; protege `u` (CH-705) → `chimulaphu`
3,+ P1 `tha` (antes C): `i` protegida: no se borra (CH-705) → `chimulanitha`,+ P1 `phi` (antes C): `u` protegida: no se borra (CH-705) → `chimulaphuphi`
4,+ AFF `wa` (antes —; decide P1 (después C)): borra `a` → `chimulanithwa`,+ AFF `yi` (antes —; decide P1 (después C)): borra `i` → `chimulaphuphyi`
salida,**chimulanithwa**,**chimulaphuphyi**


In [10]:
# 3.3 · Orden de los sufijos (§6): ¿la lista de ids es una palabra posible?
SLOTS = CONFIG["slots"]
PLANTILLAS_NOMBRE = {n: re.compile(rx) for n, rx in SLOTS["nombre"]["plantillas"].items()}
PLANTILLA_VERBO = re.compile(SLOTS["verbo"]["plantilla"])
GRUPOS_ID = {                                 # nombres de grupo que usa 'excluye' en slots.yaml
    "POSS": SLOTS["nombre"]["slots"]["N2"],
    "CASO": SLOTS["nombre"]["slots"]["N4"],
}
INTERROGATIVOS_B = {p["b"] for i, p in GRAMATICALES.items() if i.startswith("INT.")}


def errores_ids_nombre(ids, base=None):
    """Revisa la secuencia de ids de una palabra nominal (nombre, pronombre, predicado).

    Parámetros:
        ids: lista de ids de morfemas (sin la raíz).
        base: forma B de la base, opcional. Sirve para la regla de Q.INF.
    Devuelve:
        Lista de errores con su regla. Vacía si la secuencia es posible.
    Reglas: CH-601 (plantillas NP_NORMAL, ESTAR, TENER, SER y sus restricciones), CH-514, CH-906.
        La restricción "PL no va con numeral" es de la frase, no de la palabra: se revisa en la sección 8.
    Ejemplo:
        >>> errores_ids_nombre(["PL", "POSS.1", "LOC"])
        []
        >>> errores_ids_nombre(["GEN", "TOP"])
        ['CH-601: GEN no lleva nada después']
    """
    cadena = " ".join(["RAIZ_N"] + list(ids))
    cumple = [n for n, rx in PLANTILLAS_NOMBRE.items() if rx.match(cadena)]
    errores = [] if cumple else [f"CH-601: '{' '.join(ids)}' no cumple ninguna plantilla del nombre"]
    presentes = lambda grupo: any(i in GRUPOS_ID.get(grupo, [grupo]) for i in ids)
    # 'excluye' se aplica a NP_NORMAL: ESTAR, TENER y SER ya fijan su estructura (utankthwa = LOC … AFF). Ver P-03.
    for a, b in (SLOTS["nombre"]["excluye"] if "NP_NORMAL" in cumple or not cumple else []):
        if b == "CUALQUIERA_DESPUES":
            if a in ids and ids.index(a) < len(ids) - 1:
                errores.append(f"CH-601: {a} no lleva nada después")
        elif b == "NUMERAL_EN_FRASE":
            continue                                     # se revisa en la frase (sección 8)
        elif presentes(a) and presentes(b):
            errores.append(f"CH-601: {a} y {b} no van juntos")
    if "LOC" in ids and ids.index("LOC") < len(ids) - 1 and ids[ids.index("LOC") + 1] != "VBLZ.KA":
        errores.append("CH-601: tras LOC solo puede ir VBLZ.KA")   # texto de CH-601 (ver PREGUNTAS.md, P-03)
    if base is not None and "Q.INF" in ids and base not in INTERROGATIVOS_B:
        errores.append("CH-906: Q.INF solo va en interrogativos")
    return sorted(set(errores))


def errores_ids_verbo(ids):
    """Revisa la secuencia de ids de un verbo.

    Parámetros:
        ids: lista de ids de morfemas (sin la raíz).
    Devuelve:
        Lista de errores con su regla. Vacía si la secuencia es posible.
    Reglas: CH-602 (plantilla y NEG ⇔ NEG.TI), CH-603 (un solo morfema en V1), CH-604 (transiciones solo
        en SIMP), CH-605 (negación solo en SIMP y FUT), CH-606 (progresivo solo en SIMP).
        La condición "obj_persona = s" depende del verbo: se revisa en la idea (R1, sección 9).
    Ejemplo:
        >>> errores_ids_verbo(["PL.V", "REM.C", "P1"])
        []
        >>> errores_ids_verbo(["NEG", "REM.C", "P1", "NEG.TI"])
        ['CH-605: NEG solo en SIMP y FUT']
    """
    errores = []
    if not PLANTILLA_VERBO.match(" ".join(["RAIZ_V"] + list(ids))):
        errores.append(f"CH-602: '{' '.join(ids)}' no cumple la plantilla del verbo")
    if ("NEG" in ids) != ("NEG.TI" in ids):
        errores.append("CH-602: NEG y NEG.TI van siempre juntos")
    if sum(i in SLOTS["verbo"]["slots"]["V1"] for i in ids) > 1:
        errores.append("CH-603: PL.V, PROG y NEG se excluyen")
    remoto = any(i in SLOTS["verbo"]["slots"]["V4"] for i in ids)
    futuro = any(i in SLOTS["verbo"]["slots"]["V5"]["FUT"] for i in ids)
    if "NEG" in ids and remoto:
        errores.append("CH-605: NEG solo en SIMP y FUT")
    if "PROG" in ids and (remoto or futuro):
        errores.append("CH-606: PROG solo en SIMP")
    return errores


print(errores_ids_nombre(["PL", "POSS.1", "LOC"]), errores_ids_nombre(["GEN", "TOP"]))
print(errores_ids_verbo(["PL.V", "REM.C", "P1"]), errores_ids_verbo(["PL.V", "NEG", "P3", "NEG.TI"]))

[] ['CH-601: GEN no lleva nada después']
[] ["CH-602: 'PL.V NEG P3 NEG.TI' no cumple la plantilla del verbo", 'CH-603: PL.V, PROG y NEG se excluyen']


### Pruebas de la sección 3

- **§5 (tabla de morfemas):** el mapeo B → A cumple CH-501, CH-502 y CH-503.
- **§6 (orden de los sufijos):** al menos una prueba por regla CH-601 a CH-606.
- **§7 (algoritmo):** los 16 ejemplos paso a paso de §7.1 de la especificación, en los dos modos.

In [11]:
# 3.4 · Pruebas de §5, §6 y §7.
def silabas(forma):
    """Cuenta las sílabas de una forma (una por vocal, corta o larga). El '¨' no cuenta.

    Parámetros:
        forma: forma de un sufijo, como "tana" o "¨ta".
    Devuelve:
        Número de sílabas.
    Reglas: CH-503 (condición 1).
    Ejemplo:
        >>> silabas("ñäni")
        2
    """
    f = forma.lstrip("¨")
    return sum(1 for g in segmentar(f) if g in VOCALES_CORTAS or g in VOCALES_LARGAS) if f else 0


def tipo_inicio(forma):
    """Clasifica el inicio de una forma: 'V', 'C' (una consonante) o 'CC' (dos).

    Parámetros:
        forma: forma de un sufijo.
    Devuelve:
        'V', 'C', 'CC' o '' si la forma es solo '¨'.
    Reglas: CH-503 (condición 2).
    Ejemplo:
        >>> tipo_inicio("sma")
        'CC'
    """
    c = clases(segmentar(forma.lstrip("¨"))) if forma.lstrip("¨") else ""
    return "" if not c else "V" if c[0] in "VL" else "CC" if c[:2] == "CC" else "C"


# Los 16 ejemplos de §7.1: (base, ids, B, A).
EJEMPLOS_7_1 = [
    ("tisqa", "P1 AFF", "tisqthwa", "tisqphyi"),
    ("tisqa", "P2 AFF", "tisqtawa", "tisqkuyi"),
    ("tisqa", "P4 AFF", "tisqtanwa", "tisqkilyi"),
    ("ch'ixra", "T31 AFF", "ch'ixrituwa", "ch'ixruquyi"),
    ("ch'ixra", "T32 AFF", "ch'ixrtamwa", "ch'ixrkiqyi"),
    ("tisqa", "NEG FUT.1 NEG.TI", "tisqkäti", "tisqñüji"),
    ("tisqa", "NEG P1 NEG.TI", "tisqkthti", "tisqñphji"),
    ("tisqa", "NEG FUT.3 NEG.TI", "tisqkaniti", "tisqñuphuji"),
    ("tisqa", "PL.V REM.C P1", "tisqapxayätha", "tisqamtujüphi"),
    ("tisqa", "PL.V REM.C P3.REM", "tisqapxäna", "tisqamtümu"),
    ("tisqa", "REM.L P4 AFF", "tisqatätanwa", "tisqawükilyi"),
    ("tisqa", "PROG P3 AFF", "tisqaskiwa", "tisqanchuyi"),
    ("chimula", "LOC VBLZ.KA P1 AFF", "chimulankthwa", "chimulamñphyi"),
    ("chimula", "POSR VBLZ.LONG P1 AFF", "chimulanithwa", "chimulaphuphyi"),
    ("jumpu", "VBLZ.LONG P1 AFF", "jumpüthwa", "jumpüphyi"),
    ("chimula", "PL POSS.1 LOC", "chimulanakajana", "chimulasululumu"),
]


class PruebasMorfemas(unittest.TestCase):
    def test_CH501_mapeo_por_forma(self):
        """CH-501: una cadena B da siempre la misma cadena A (homófonos de la tabla)."""
        a_de_b = {}
        for m in MORFEMAS.values():
            a_de_b.setdefault(m["b"], set()).add(m["a"])
        self.assertEqual({b: a for b, a in a_de_b.items() if len(a) > 1}, {})
        homofonos = {"ta": "ku", "sa": "khu", "ni": "phu", "na": "mu", "ka": "ñu", "ti": "ji", "ru": "llu"}
        for b, a in homofonos.items():
            self.assertEqual(a_de_b[b], {a}, b)
        for b, a in ((m["b"], m["a"]) for m in MORFEMAS.values()):   # ¨ta → ¨ + A(ta)
            if b.startswith("¨") and b[1:] in a_de_b:
                self.assertEqual(a, "¨" + next(iter(a_de_b[b[1:]])), b)

    def test_CH502_alargamiento_igual_en_A(self):
        """CH-502: los morfemas que son solo '¨' son iguales en A; tisqa + FUT.1 = tisqä en los dos modos."""
        for m in MORFEMAS.values():
            if m["b"] == "¨":
                self.assertEqual(m["a"], "¨", m["id"])
        self.assertEqual(construir_ambos("tisqa", ["FUT.1"]), ("tisqä", "tisqä"))

    def test_CH503_condiciones_de_las_formas_A(self):
        """CH-503: mismas sílabas, mismo tipo de inicio, larga en la misma sílaba, alfabeto y B distintas → A distintas."""
        for m in MORFEMAS.values():
            b, a = m["b"], m["a"]
            with self.subTest(morfema=m["id"]):
                segmentar(a.lstrip("¨") or "a")                                  # condición 4
                self.assertEqual(silabas(b), silabas(a))                         # condición 1
                self.assertEqual(tipo_inicio(b), tipo_inicio(a))                 # condición 2
                larga = lambda f: [g in VOCALES_LARGAS for g in segmentar(f.lstrip("¨")) if g in VOCALES_CORTAS | VOCALES_LARGAS] if f.lstrip("¨") else []
                self.assertEqual(larga(b), larga(a))                             # condición 3
        b_de_a = {}
        for m in MORFEMAS.values():
            b_de_a.setdefault(m["a"], set()).add(m["b"])
        self.assertEqual({a: b for a, b in b_de_a.items() if len(b) > 1}, {})   # condición 6
        formas_b = {m["b"] for m in MORFEMAS.values()} - {"¨"}
        self.assertEqual({m["a"] for m in MORFEMAS.values()} & formas_b, set())  # condición 5 (contra los sufijos B)

    def test_CH505_palabras_gramaticales(self):
        """CH-505: cada palabra gramatical tiene forma B y A con el mismo número de sílabas, y A ≠ B."""
        for p in GRAMATICALES.values():
            with self.subTest(palabra=p["id"]):
                self.assertEqual(silabas(p["b"]), silabas(p["a"]))
                self.assertNotEqual(p["b"], p["a"])
        self.assertEqual(construir_ambos("jiwasa", ["PL", "TOP"]), ("jiwasanakaxa", "wiqutusuluxu"))   # CH-708

    def test_CH511_remoto_tercera(self):
        """CH-511: REM.C toma -¨- ante P3.REM (tisqäna / tisqämu)."""
        self.assertEqual(construir_ambos("tisqa", ["REM.C", "P1"]), ("tisqayätha", "tisqajüphi"))
        self.assertEqual(construir_ambos("tisqa", ["REM.C", "P3.REM"]), ("tisqäna", "tisqämu"))

    def test_CH512_rem_l_con_ta(self):
        """CH-512: REM.L usa -tä- (tisqatäna / tisqawümu), nunca -tayna."""
        self.assertEqual(construir_ambos("tisqa", ["REM.L", "P3.REM"]), ("tisqatäna", "tisqawümu"))

    def test_CH513_futuro_primera(self):
        """CH-513: FUT.1 = -¨ (tisqä)."""
        self.assertEqual(construir_palabra("tisqa", ["FUT.1"]), "tisqä")


class PruebasSlots(unittest.TestCase):
    def test_CH601_plantilla_nombre(self):
        """CH-601: plantillas NP_NORMAL, ESTAR, TENER y SER, y sus restricciones."""
        for ids in ["PL POSS.1 LOC", "LOC VBLZ.KA P1 AFF", "PL POSR VBLZ.LONG P1 AFF", "POSR AFF", "VBLZ.LONG P2 Q.YN", "COM AFF"]:
            self.assertEqual(errores_ids_nombre(ids.split()), [], ids)
        for ids in ["POSS.1 POSR", "POSR LOC", "UNTIL PL", "PL UNTIL", "POSS.2 UNTIL", "GEN TOP",
                    "ABL AFF", "LOC AFF", "LOC Q.INF", "LOC TOP", "VBLZ.LONG P3 AFF", "POSR VBLZ.LONG P3", "LOC PL"]:
            self.assertNotEqual(errores_ids_nombre(ids.split()), [], ids)
        self.assertEqual(errores_ids_nombre(["Q.INF"], base="khiti"), [])
        self.assertTrue(tiene_regla(errores_ids_nombre(["Q.INF"], base="naya"), "CH-906"))

    def test_CH602_plantilla_verbo(self):
        """CH-602: tisqa-pxa-yä-tha es válida; NEG sin NEG.TI no."""
        self.assertEqual(errores_ids_verbo(["PL.V", "REM.C", "P1"]), [])
        self.assertEqual(errores_ids_verbo(["NEG", "FUT.3", "NEG.TI"]), [])
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "P3"]), "CH-602"))
        self.assertTrue(tiene_regla(errores_ids_verbo(["P3", "P1"]), "CH-602"))

    def test_CH603_v1_un_solo_morfema(self):
        """CH-603: PL.V + NEG es inválido (tisqa-pxa-ka-i-ti)."""
        self.assertNotEqual(errores_ids_verbo(["PL.V", "NEG", "P3", "NEG.TI"]), [])
        self.assertNotEqual(errores_ids_verbo(["PL.V", "PROG", "P3"]), [])

    def test_CH604_transiciones_solo_simp(self):
        """CH-604: no hay transición en los remotos ni en el futuro (REM.C + T31)."""
        self.assertEqual(errores_ids_verbo(["T31", "AFF"]), [])
        self.assertNotEqual(errores_ids_verbo(["REM.C", "T31"]), [])
        self.assertNotEqual(errores_ids_verbo(["T31", "FUT.1"]), [])

    def test_CH605_negacion_simp_fut(self):
        """CH-605: NEG + REM.C o REM.L es inválido."""
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "REM.C", "P1", "NEG.TI"]), "CH-605"))
        self.assertTrue(tiene_regla(errores_ids_verbo(["NEG", "REM.L", "P3.REM", "NEG.TI"]), "CH-605"))

    def test_CH606_progresivo_simp(self):
        """CH-606: PROG + FUT es inválido; PROG + P3 es válido."""
        self.assertEqual(errores_ids_verbo(["PROG", "P3", "AFF"]), [])
        self.assertTrue(tiene_regla(errores_ids_verbo(["PROG", "FUT.3"]), "CH-606"))


class PruebasMotor(unittest.TestCase):
    def test_CH701_ejemplos_paso_a_paso(self):
        """CH-701: los 16 ejemplos de §7.1 dan la salida de la especificación en B y en A."""
        for base, ids, b, a in EJEMPLOS_7_1:
            with self.subTest(base=base, ids=ids):
                self.assertEqual(construir_ambos(base, ids.split()), (b, a))

    def test_CH702_oracional_no_decide(self):
        """CH-702: tras P1 (después C) el oracional borra; tras P2 (después V) conserva."""
        self.assertEqual(construir_palabra("tisqa", ["P1", "AFF"]), "tisqthwa")
        self.assertEqual(construir_palabra("tisqa", ["P2", "AFF"]), "tisqtawa")

    def test_CH703_la_larga_no_se_borra(self):
        """CH-703: tisqa-yä-tha → tisqayätha (P1 pide borrar, pero ä es larga)."""
        _, _, dec = construir_palabra("tisqa", ["REM.C", "P1"], traza=True)
        self.assertEqual(dec[-1], "larga-no-borra")

    def test_CH704_alargamiento(self):
        """CH-704: tisqa-¨ta → tisqäta (B) / tisqäku (A)."""
        self.assertEqual(construir_ambos("tisqa", ["FUT.2"]), ("tisqäta", "tisqäku"))

    def test_CH705_vblz_long_tras_posr(self):
        """CH-705: chimula-ni-(¨)-tha-wa → chimulanithwa; la decisión es 'proteger'."""
        _, _, dec = construir_palabra("chimula", ["POSR", "VBLZ.LONG", "P1", "AFF"], traza=True)
        self.assertEqual(dec, ["conservar", "proteger", "protegida-no-borra", "borrar"])

    def test_CH706_grupos_enteros(self):
        """CH-706: tisqkthti y tisqñphji conservan todo el grupo de consonantes."""
        self.assertEqual(construir_ambos("tisqa", ["NEG", "P1", "NEG.TI"]), ("tisqkthti", "tisqñphji"))

    def test_CH708_gramaticales_en_A(self):
        """CH-708: en A, la palabra gramatical cambia antes de añadir sufijos (naya → muqi)."""
        self.assertEqual(construir_ambos("naya", ["TOP"]), ("nayaxa", "muqixu"))
        self.assertEqual(construir_ambos("jani", ["AFF"]), ("janiwa", "yuliyi"))

    def test_CH709_palabra_invalida_es_error(self):
        """CH-709: una palabra inválida es un error del programa (xiku empieza por x)."""
        with self.assertRaises(ValueError):
            construir_palabra("xiku", [])


correr_pruebas(PruebasMorfemas, PruebasSlots, PruebasMotor, titulo="motor morfológico (§5, §6, §7)")

test_CH501_mapeo_por_forma (__main__.PruebasMorfemas.test_CH501_mapeo_por_forma)
CH-501: una cadena B da siempre la misma cadena A (homófonos de la tabla). ... 

ok


test_CH502_alargamiento_igual_en_A (__main__.PruebasMorfemas.test_CH502_alargamiento_igual_en_A)
CH-502: los morfemas que son solo '¨' son iguales en A; tisqa + FUT.1 = tisqä en los dos modos. ... 

ok


test_CH503_condiciones_de_las_formas_A (__main__.PruebasMorfemas.test_CH503_condiciones_de_las_formas_A)
CH-503: mismas sílabas, mismo tipo de inicio, larga en la misma sílaba, alfabeto y B distintas → A distintas. ... 

ok


... (34 outputs left out)

test_CH709_palabra_invalida_es_error (__main__.PruebasMotor.test_CH709_palabra_invalida_es_error)
CH-709: una palabra inválida es un error del programa (xiku empieza por x). ... 

ok


----------------------------------------------------------------------
Ran 21 tests in 0.009s

OK


✔ motor morfológico (§5, §6, §7): 21 pruebas en verde.


## 4. Pruebas de referencia (CH-710)

**Qué hace esta sección.** Prueba el motor con **raíces reales** del aymara. Si el motor reproduce las formas de los libros, sus reglas son correctas.

1. **Prueba de igualdad (modo B).** Los 102 casos de `regresion_ch710.json`. Cada caso es una raíz real, una lista de ids y la forma que aparece en el libro L2. El motor debe dar **exactamente** esa forma.
2. **Prueba de paralelismo (modo A).** Para cada caso, el motor también construye la palabra en modo A. En cada paso, la decisión (borrar, conservar, alargar o proteger) debe ser la misma que en B. Así sabemos que A es un "espejo" fiel de B.
3. **Paradigmas (§8).** Las 64 celdas de `paradigmas.yaml` con el verbo de ejemplo `ch'ixra` "ver", en B y en A.

Los datos se usan **tal como están**, sin cambios.

Si un solo caso falla, el notebook se detiene: alguna regla está mal y no se deben generar datos.

**Dónde están los 40 casos de CH-1201.** Necesitan el constructor de oraciones y el español. Por eso se prueban al final de la sección 8. La sección 10 repite todas las pruebas de referencia antes de generar (CH-1201: "si un caso falla, la generación no se hace").

**Reglas que implementa.** CH-710 (los tres puntos) y las tablas de §8.

**Qué entra.** `regresion_ch710.json` y el bloque `paradigmas.yaml`. **Qué sale.** Una tabla de resultados.

**Ejemplo** (caso de CH-710): `sara` + `PL.V REM.C P1`:
- B: `sarapxayätha` (L2 p.181) ✓ · A: `saramtujüphi` · Español: "hemos ido".
- Decisiones en los dos modos: conservar, conservar, larga-no-borra.

In [12]:
# 4.1 · Cargar los 102 casos de CH-710 y compararlos con el bloque de la especificación.
REGRESION = json.loads((CARPETA_DATOS / "regresion_ch710.json").read_text(encoding="utf-8"))
clave_caso = lambda c: (c["raiz"], tuple(c["ids"]), c["esperado"])
if "regresion_raices_reales" in CONFIG:
    en_spec = {clave_caso(c) for c in CONFIG["regresion_raices_reales"]["casos"]}
    distintos = en_spec ^ {clave_caso(c) for c in REGRESION}
    print("El JSON y el bloque CH-710 de la especificación", "coinciden." if not distintos else
          f"NO coinciden en {len(distintos)} casos (se usa el JSON): {sorted(distintos)[:5]}")
print(f"{len(REGRESION)} casos de regresión cargados.")


def resultado_caso(c):
    """Construye un caso de CH-710 en los dos modos y compara con lo esperado.

    Parámetros:
        c: caso del JSON, con 'raiz', 'ids', 'esperado' y 'fuente'.
    Devuelve:
        Diccionario con la salida en B y en A, si B es igual a lo esperado y si las decisiones coinciden.
    Reglas: CH-710.1 (igualdad) y CH-710.2 (paralelismo).
    Ejemplo:
        >>> resultado_caso({"raiz": "sara", "ids": ["P1", "AFF"], "esperado": "sarthwa", "fuente": "L2 p.176"})["igual"]
        True
    """
    b, _, dec_b = construir_palabra(c["raiz"], c["ids"], "B", traza=True)
    a, _, dec_a = construir_palabra(c["raiz"], c["ids"], "A", traza=True)
    return {"raíz": c["raiz"], "ids": " ".join(c["ids"]), "esperado": c["esperado"], "B": b, "A": a,
            "igual": b == c["esperado"], "paralelo": dec_b == dec_a, "decisiones": ", ".join(dec_b),
            "fuente": c["fuente"]}


TABLA_CH710 = pd.DataFrame([resultado_caso(c) for c in REGRESION])
display(TABLA_CH710.sample(12, random_state=0).drop(columns="fuente"))

El JSON y el bloque CH-710 de la especificación coinciden.
102 casos de regresión cargados.


,raíz,ids,esperado,B,A,igual,paralelo,decisiones
26,sara,FUT.3,sarani,sarani,saraphu,True,True,conservar
92,jiwasa,TOP,jiwasaxa,jiwasaxa,wiqutuxu,True,True,conservar
2,manq'a,P4 AFF,manq'tanwa,manq'tanwa,manq'kilyi,True,True,"borrar, borrar"
55,sara,PROG P3,saraski,saraski,saranchu,True,True,"conservar, borrar"
91,juma,TOP,jumaxa,jumaxa,tiluxu,True,True,conservar
84,marka,UNTIL AFF,markakamawa,markakamawa,markachiwiyi,True,True,"conservar, conservar"
16,manq'a,P3,manq'i,manq'i,manq'u,True,True,borrar
66,phuchha,POSS.4,phuchhasa,phuchhasa,phuchhakhu,True,True,conservar
54,sara,PL.V REM.C P3.REM,sarapxäna,sarapxäna,saramtümu,True,True,"conservar, alargar, conservar"
68,uta,PL POSS.1 LOC,utanakajana,utanakajana,utasululumu,True,True,"conservar, conservar, conservar"


In [13]:
# 4.2 · Pruebas de referencia. Si una falla, el notebook se detiene.
class PruebasReferencia(unittest.TestCase):
    def test_CH710_cantidad(self):
        """CH-710: la lista tiene 102 casos."""
        self.assertEqual(len(REGRESION), 102)

    def test_CH710_1_igualdad_modo_B(self):
        """CH-710.1: el modo B reproduce exactamente las 102 formas de la guía."""
        fallas = TABLA_CH710[~TABLA_CH710["igual"]][["raíz", "ids", "esperado", "B", "fuente"]]
        self.assertTrue(fallas.empty, "\n" + fallas.to_string())

    def test_CH710_2_paralelismo_modo_A(self):
        """CH-710.2: en los 102 casos, el modo A toma las mismas decisiones que el B."""
        fallas = TABLA_CH710[~TABLA_CH710["paralelo"]][["raíz", "ids", "B", "A"]]
        self.assertTrue(fallas.empty, "\n" + fallas.to_string())

    def test_CH710_ejemplo(self):
        """CH-710: sara + PL.V REM.C P1 → sarapxayätha / saramtujüphi, con decisiones [conservar, conservar, larga-no-borra]."""
        b, _, dec = construir_palabra("sara", ["PL.V", "REM.C", "P1"], "B", traza=True)
        self.assertEqual((b, construir_palabra("sara", ["PL.V", "REM.C", "P1"], "A")), ("sarapxayätha", "saramtujüphi"))
        self.assertEqual(dec, ["conservar", "conservar", "larga-no-borra"])

    def test_paradigmas_seccion_8(self):
        """§8: las celdas de paradigmas.yaml dan las formas de prueba en B y en A."""
        P = CONFIG["paradigmas"]
        for celda in P["celdas"]:
            with self.subTest(tiempo=celda["tiempo"], celda=celda["celda"], ids=celda["ids"]):
                self.assertEqual(construir_ambos(P["verbo_prueba"], celda["ids"]),
                                 (celda["prueba"]["b"], celda["prueba"]["a"]))
        print(f"\n   ({len(P['celdas'])} celdas de paradigma probadas)", end=" ")


correr_pruebas(PruebasReferencia, titulo="pruebas de referencia CH-710")
resumen = pd.DataFrame([
    {"prueba": "CH-710.1 igualdad (modo B)", "casos": len(TABLA_CH710), "en verde": int(TABLA_CH710["igual"].sum())},
    {"prueba": "CH-710.2 paralelismo (modo A)", "casos": len(TABLA_CH710), "en verde": int(TABLA_CH710["paralelo"].sum())},
    {"prueba": "§8 paradigmas (B y A)", "casos": len(CONFIG["paradigmas"]["celdas"]), "en verde": len(CONFIG["paradigmas"]["celdas"])},
])
display(resumen)

test_CH710_1_igualdad_modo_B (__main__.PruebasReferencia.test_CH710_1_igualdad_modo_B)
CH-710.1: el modo B reproduce exactamente las 102 formas de la guía. ... 

ok


test_CH710_2_paralelismo_modo_A (__main__.PruebasReferencia.test_CH710_2_paralelismo_modo_A)
CH-710.2: en los 102 casos, el modo A toma las mismas decisiones que el B. ... 

ok


test_CH710_cantidad (__main__.PruebasReferencia.test_CH710_cantidad)
CH-710: la lista tiene 102 casos. ... 

ok


... (3 outputs left out)


   (64 celdas de paradigma probadas) ok


----------------------------------------------------------------------
Ran 5 tests in 0.005s

OK


✔ pruebas de referencia CH-710: 5 pruebas en verde.


,prueba,casos,en verde
0,CH-710.1 igualdad (modo B),102,102
1,CH-710.2 paralelismo (modo A),102,102
2,§8 paradigmas (B y A),64,64


## 5. Léxico

**Qué hace esta sección.** Lee el **léxico**: la lista de palabras con significado (nombres, verbos, adjetivos y adverbios de tiempo). Cada fila es un **sentido** del español ("niño", "ir"…) con sus datos.

**De dónde sale.**
- Si en `datos/` está `lexico_sin_raices.csv` (lo entrega el rol 1), se usa ese archivo. Sus raíces las fabrica la sección 6.
- Si no está, se usan las **25 entradas de ejemplo** de §4.1 de la especificación. Esas ya traen raíz: la sección 6 solo las revisa (ver PREGUNTAS.md, P-05).

**Qué se revisa** (CH-402 y CH-406). Que estén todas las columnas, que cada valor sea uno de los permitidos y que las listas de clases sean coherentes. Por ejemplo: un `N_HUM` siempre tiene `clase_sem = persona`, y un objeto directo nunca es `persona` (CH-914).

**Términos.**
- **categoría** (`cat`): tipo de palabra. `N_HUM` = nombre de persona, `V_TR` = verbo transitivo, etc. (CH-401).
- **clase semántica** (`clase_sem`): grupo de significado de un nombre: `comida`, `animal`, `lugar`… (CH-406).
- **rango**: puesto de frecuencia dentro de la categoría. 1 = la palabra más común (CH-403).

**Qué entra.** `lexico_sin_raices.csv` o el bloque `csv` de §4.1. **Qué sale.** La tabla `LEXICO`.

**Ejemplo** (fila L0013 de §4.1): "ir" es `V_MOV`, su sujeto es `persona` o `animal` y no admite LOC (`NINGUNA`). Su raíz es `tisqa`:
- B: `tisqthwa` — ir-1>3.SIMP-AFF — "voy".
- A: `tisqphyi`.

In [14]:
# 5.1 · Leer el léxico y revisar su formato (CH-401, CH-402, CH-406).
import io

CATEGORIAS = list(CONFIG["generador_raices"]["silabas"])          # CH-401: las 8 categorías
VERBOS = [c for c in CATEGORIAS if c.startswith("V_")]
NOMBRES = ["N_HUM", "N_NOHUM"]
CLASES_SEM = ["persona", "animal", "comida", "bebida", "ropa", "herramienta", "recipiente",
              "planta", "lugar", "vivienda", "tiempo", "abstracto", "objeto"]   # tabla 1 de CH-406
SUBCAT = {"ADJ": ["tamano", "color", "otro"], "ADV_T": ["pasado", "presente", "futuro"]}   # CH-401

CSV_EJEMPLO = re.search(r"```csv\n(.*?)```", TEXTO_SPEC, re.S).group(1)   # §4.1
COLUMNAS_LEXICO = CSV_EJEMPLO.splitlines()[0].split(",")                   # CH-402


def leer_csv(fuente):
    """Lee un léxico en formato CH-402. Todo se lee como texto: "-" y las celdas vacías se conservan.

    Parámetros:
        fuente: ruta del archivo o texto csv (io.StringIO).
    Devuelve:
        DataFrame con una fila por sentido.
    Reglas: CH-402 (UTF-8, separador ",").
    Ejemplo:
        >>> leer_csv(io.StringIO(CSV_EJEMPLO)).shape
        (25, 21)
    """
    df = pd.read_csv(fuente, dtype=str, keep_default_na=False, encoding="utf-8")
    return df.apply(lambda col: col.map(lambda v: unicodedata.normalize("NFC", v)))   # CH-402: NFC


def errores_lexico(df):
    """Revisa el formato y la coherencia de un léxico.

    Parámetros:
        df: DataFrame que devuelve leer_csv().
    Devuelve:
        Lista de errores, cada uno con el id de la fila y su regla. Vacía si todo está bien.
    Reglas: CH-401 (categorías y subcategorías), CH-402 (columnas y valores), CH-405 (género),
        CH-406 (clases), CH-909 (rige_ru solo en V_DITR), CH-914 (el OD nunca es persona).
    Ejemplo:
        >>> errores_lexico(LEXICO_EJEMPLO)
        []
    """
    faltan = [c for c in COLUMNAS_LEXICO if c not in df.columns]
    if faltan:
        return [f"CH-402: faltan columnas {faltan}"]
    errores = [f"CH-402: id repetido {i}" for i in df["id"][df["id"].duplicated()]]

    def lista(v):
        return v.split("|") if v not in ("-", "", "NINGUNA") else []

    for f in df.itertuples():
        e = []
        cat = f.cat
        if not re.fullmatch(r"L\d{4}", f.id):
            e.append("CH-402: id con formato distinto de L0001")
        if cat not in CATEGORIAS:
            errores.append(f"{f.id}: CH-401: categoría desconocida {cat!r}")
            continue
        if f.subcat not in SUBCAT.get(cat, ["-"]):
            e.append(f"CH-401: subcat {f.subcat!r} no vale para {cat}")
        es_nombre, es_verbo, con_od = cat in NOMBRES, cat in VERBOS, cat in ("V_TR", "V_DITR")
        clase_ok = ("persona",) if cat == "N_HUM" else CLASES_SEM[1:] if cat == "N_NOHUM" else ("-",)
        if f.clase_sem not in clase_ok:
            e.append(f"CH-406: clase_sem {f.clase_sem!r} no vale para {cat}")
        if not f.es_m:
            e.append("CH-402: es_m vacío")
        if f.genero not in (["m", "f", "var"] if es_nombre else ["-"]):
            e.append(f"CH-405: genero {f.genero!r} no vale para {cat}")
        if f.genero == "var" and not f.es_f:
            e.append("CH-405: genero = var sin forma femenina (es_f)")
        if f.contable not in (["s", "n"] if es_nombre else ["-"]):
            e.append(f"CH-402: contable {f.contable!r} no vale para {cat}")
        if f.rige_ru not in ((["s", "n"] if cat == "V_DITR" else ["n"]) if es_verbo else ["-"]):
            e.append(f"CH-909: rige_ru {f.rige_ru!r} no vale para {cat}")
        if f.obj_persona not in (["s", "n"] if con_od else ["-"]):
            e.append(f"CH-604: obj_persona {f.obj_persona!r} no vale para {cat}")
        if not f.rango.isdigit() or int(f.rango) < 1:
            e.append(f"CH-403: rango {f.rango!r} no es un entero ≥ 1")
        # Listas de clases: obligatorias u opcionales según CH-402; con clases de CH-406.
        obligatorias = {"suj_clases": es_verbo, "obj_clases": con_od, "mod_clases": cat == "ADJ"}
        permitidas = {"suj_clases": es_verbo, "obj_clases": con_od, "instr_clases": es_verbo,
                      "loc_clases": es_verbo, "mod_clases": cat == "ADJ"}
        for col, permitida in permitidas.items():
            v = getattr(f, col)
            if not permitida and v != "-":
                e.append(f"CH-402: {col} debe ser '-' en {cat}")
            elif obligatorias.get(col) and not lista(v):
                e.append(f"CH-402: {col} es obligatoria en {cat}")
            elif v == "NINGUNA" and col != "loc_clases":
                e.append(f"CH-402: NINGUNA solo vale en loc_clases")
            otras = [c for c in lista(v) if c not in CLASES_SEM]
            if otras:
                e.append(f"CH-406: {col} con clases desconocidas {otras}")
        if "persona" in lista(f.obj_clases):
            e.append("CH-914: el objeto directo nunca es persona")
        if cat == "V_MOV" and f.loc_clases != "NINGUNA":
            e.append("CH-406: los V_MOV llevan siempre loc_clases = NINGUNA")
        errores += [f"{f.id}: {x}" for x in e]
    rep = df[df.duplicated(["cat", "rango"], keep=False)]
    errores += [f"{i}: CH-403: rango repetido dentro de {c}" for i, c in zip(rep["id"], rep["cat"])]
    return errores


LEXICO_EJEMPLO = leer_csv(io.StringIO(CSV_EJEMPLO))
RUTA_LEXICO_PROD = CARPETA_DATOS / "lexico_sin_raices.csv"
LEXICO_PRODUCCION = RUTA_LEXICO_PROD.exists()                  # P-05
LEXICO = leer_csv(RUTA_LEXICO_PROD) if LEXICO_PRODUCCION else LEXICO_EJEMPLO.copy()
if LEXICO_PRODUCCION:                                         # el rol 1 entrega el archivo sin raíz ni sílabas
    LEXICO = LEXICO.assign(raiz="", silabas="").reindex(columns=COLUMNAS_LEXICO)

errores = errores_lexico(LEXICO)
if errores:
    raise ValueError(f"El léxico tiene {len(errores)} errores:\n" + "\n".join(errores[:50]))
print(f"Léxico: {'lexico_sin_raices.csv' if LEXICO_PRODUCCION else '§4.1 de la especificación (ejemplo, P-05)'}"
      f" · {len(LEXICO)} entradas · sin errores de formato.")
display(LEXICO.groupby("cat").size().reindex(CATEGORIAS, fill_value=0).rename("entradas").to_frame().T)

Léxico: §4.1 de la especificación (ejemplo, P-05) · 25 entradas · sin errores de formato.


cat,N_HUM,N_NOHUM,V_INTR,V_MOV,V_TR,V_DITR,ADJ,ADV_T
entradas,4,8,1,2,4,1,2,3


### Pruebas de la sección 5

- **CH-402:** el léxico de §4.1 tiene las 21 columnas y ningún error.
- **CH-401, CH-406, CH-909, CH-914:** si se estropea una fila a propósito, la revisión lo detecta y cita la regla.

In [15]:
# 5.2 · Pruebas del léxico.
class PruebasLexico(unittest.TestCase):
    def roto(self, i, **cambios):
        """Copia del léxico de §4.1 con la fila i cambiada."""
        df = LEXICO_EJEMPLO.copy()
        for col, v in cambios.items():
            df.loc[i, col] = v
        return errores_lexico(df)

    def test_CH402_ejemplo_correcto(self):
        """CH-402: las 25 entradas de §4.1 tienen las 21 columnas y pasan la revisión."""
        self.assertEqual(list(LEXICO_EJEMPLO.columns), COLUMNAS_LEXICO)
        self.assertEqual((len(LEXICO_EJEMPLO), len(COLUMNAS_LEXICO)), (25, 21))
        self.assertEqual(errores_lexico(LEXICO_EJEMPLO), [])

    def test_CH402_falta_columna(self):
        """CH-402: sin la columna rango, el léxico no se acepta."""
        self.assertTrue(tiene_regla(errores_lexico(LEXICO_EJEMPLO.drop(columns="rango")), "CH-402"))

    def test_CH402_id_repetido(self):
        """CH-402: dos filas con el mismo id."""
        self.assertTrue(tiene_regla(self.roto(1, id="L0001"), "CH-402"))

    def test_CH401_categoria_desconocida(self):
        """CH-401: 'PRON' no es una categoría del léxico (los pronombres son palabras gramaticales)."""
        self.assertTrue(tiene_regla(self.roto(0, cat="PRON"), "CH-401"))

    def test_CH406_nhum_es_persona(self):
        """CH-406: un N_HUM ("niño", fila 0) con clase animal."""
        self.assertTrue(tiene_regla(self.roto(0, clase_sem="animal"), "CH-406"))

    def test_CH914_od_no_persona(self):
        """CH-914: "comer" (fila 15) con persona entre las clases del OD."""
        self.assertTrue(tiene_regla(self.roto(15, obj_clases="comida|persona"), "CH-914"))

    def test_CH909_rige_ru_solo_ditr(self):
        """CH-909: "comer" (V_TR) con rige_ru = s."""
        self.assertTrue(tiene_regla(self.roto(15, rige_ru="s"), "CH-909"))

    def test_CH406_vmov_sin_loc(self):
        """CH-406: "ir" (V_MOV, fila 12) que acepte LOC."""
        self.assertTrue(tiene_regla(self.roto(12, loc_clases="-"), "CH-406"))

    def test_CH406_clase_desconocida(self):
        """CH-406: "grande" (fila 20) que modifique la clase inexistente 'mueble'."""
        self.assertTrue(tiene_regla(self.roto(20, mod_clases="objeto|mueble"), "CH-406"))


correr_pruebas(PruebasLexico, titulo="léxico (§4)")

test_CH401_categoria_desconocida (__main__.PruebasLexico.test_CH401_categoria_desconocida)
CH-401: 'PRON' no es una categoría del léxico (los pronombres son palabras gramaticales). ... 

ok


test_CH402_ejemplo_correcto (__main__.PruebasLexico.test_CH402_ejemplo_correcto)
CH-402: las 25 entradas de §4.1 tienen las 21 columnas y pasan la revisión. ... 

ok


test_CH402_falta_columna (__main__.PruebasLexico.test_CH402_falta_columna)
CH-402: sin la columna rango, el léxico no se acepta. ... 

ok


... (10 outputs left out)

test_CH914_od_no_persona (__main__.PruebasLexico.test_CH914_od_no_persona)
CH-914: "comer" (fila 15) con persona entre las clases del OD. ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.010s

OK


✔ léxico (§4): 9 pruebas en verde.


## 6. Raíces inventadas

**Qué hace esta sección.** Fabrica una **raíz inventada** para cada sentido del léxico y escribe `lexico.csv`. Corre una sola vez, con una semilla fija: así el diccionario siempre sale igual (CH-309). Las oraciones nunca inventan raíces: solo las leen de `lexico.csv`.

**Cómo se fabrica una raíz** (§3.1 de la especificación):
1. Se sortea el número de sílabas: 2 o 3, según la categoría (CH-301).
2. Se sortea la forma de cada sílaba (V, CV, VC o CVC) y sus letras, con los pesos de CH-302 y CH-303. La primera sílaba no empieza por `x` ni `r`.
3. Si es verbo, la última vocal es `a` (CH-304).
4. A veces, una consonante como `ch` se vuelve `chh` o `ch'` (**laringalización**, CH-305).
5. La raíz pasa por los **filtros**. Si falla uno, se descarta y se sortea otra (hasta 1.000 veces).

**Los filtros.**

| Regla | Se descarta la raíz si… | Ejemplo descartado |
|---|---|---|
| CH-201 a CH-209 | no cumple las reglas de sonidos de la raíz | `tissa` (s + s) |
| CH-306 | es una palabra real del aymara, o empieza igual que palabras reales | `sara` |
| CH-307 | es igual a un sufijo o a una palabra gramatical | `muqi`, `naya` |
| CH-308 | termina en un sufijo de 3 letras o más | `kurnaka` (-naka) |
| CH-309 | ya la tiene otro sentido | — |
| CH-310 | es otra raíz + un sufijo, o al revés | `kirpuru` (= `kirpu` + `-ru`) |

**El filtro de palabras reales (CH-306).** Usa `palabras_reales.txt`: una palabra aymara real por línea. Cuenta cuántas palabras reales **empiezan** por la raíz:
- Modo `"guia"`: se descarta si hay 1 o más.
- Modo `"corpus"`: se descarta si hay 5 o más.

El modo se elige en la celda de parámetros (`MODO_FILTRO`). Si el archivo falta, se avisa y las raíces se generan sin este filtro.

**Léxico de ejemplo.** Las raíces de §4.1 ya están fijadas por la especificación. No se regeneran: se revisan con los mismos filtros (P-05). Con `lexico_sin_raices.csv` sí se generan.

**Qué entra.** `LEXICO`, `palabras_reales.txt`, el bloque `generador_raices.yaml`. **Qué sale.** `salida/lexico.csv`.

**Ejemplo** ("ir", verbo): 2 sílabas, [CVC, CV] → `tis` + `qa` → `tisqa`. Pasa todos los filtros:
- B: `tisq-th-wa` → `tisqthwa` — ir-1>3.SIMP-AFF — "voy".
- A: `tisq-ph-yi` → `tisqphyi`.

In [16]:
# 6.1 · Palabras reales (CH-306) y listas de formas prohibidas (CH-307, CH-308, CH-310).
import bisect

GR = CONFIG["generador_raices"]
FR = GR["filtro_real"]
LETRAS = "".join(sorted(set("".join(GRAFEMAS))))
RE_NO_LETRA = re.compile(f"[^{re.escape(LETRAS + APOSTROFO)}]+")


def tokens_reales(texto):
    """Saca las palabras de un texto, normalizadas como las raíces.

    Parámetros:
        texto: contenido de palabras_reales.txt (o cualquier texto aymara).
    Devuelve:
        Conjunto de palabras distintas.
    Reglas: CH-306.2 (NFC, minúsculas, apóstrofos de CH-102; se corta en todo lo que no es letra ni ').
    Ejemplo:
        >>> sorted(tokens_reales("Sara-ña, ch’uxña"))
        ["ch'uxña", 'sara', 'ña']
    """
    return set(RE_NO_LETRA.split(normalizar(texto))) - {""}


RUTA_REALES = CARPETA_DATOS / FR["archivo"]
FILTRO_REAL_ACTIVO = RUTA_REALES.exists()
REALES = tokens_reales(RUTA_REALES.read_text(encoding="utf-8")) if FILTRO_REAL_ACTIVO else set()
REALES_ORDENADAS = sorted(REALES)
UMBRAL_REAL = FR["umbral_prefijo"]


def n_prefijo(r):
    """Cuenta las palabras reales distintas que empiezan por r (r incluida).

    Hace la misma cuenta que el trie de CH-306.4, con búsqueda binaria en la lista ordenada.

    Parámetros:
        r: raíz candidata.
    Devuelve:
        n(r), un entero.
    Reglas: CH-306.3, CH-306.4.
    Ejemplo:
        >>> n_prefijo("lapi")
        3
    """
    return bisect.bisect_left(REALES_ORDENADAS, r + "￿") - bisect.bisect_left(REALES_ORDENADAS, r)


def motivo_real(r, modo=None):
    """Dice si el filtro de palabras reales descarta una raíz.

    Parámetros:
        r: raíz candidata.
        modo: "guia" o "corpus". Si se omite, el de MODO_FILTRO.
    Devuelve:
        Texto con el motivo y la regla, o None si la raíz pasa.
    Reglas: CH-306.3 (guía: n(r) ≥ 1; corpus: n(r) ≥ 5; siempre se descarta una palabra real).
    Ejemplo:
        >>> motivo_real("sara")
        'CH-306: es una palabra real'
    """
    modo = modo or FR["modo"]
    if FR["descartar_si_igual"] and r in REALES:
        return "CH-306: es una palabra real"
    n = n_prefijo(r)
    if n >= UMBRAL_REAL[modo]:
        return f"CH-306: {n} palabras reales empiezan por {r} (modo {modo}, umbral {UMBRAL_REAL[modo]})"
    return None


# CH-307: formas de sufijos (también el alomorfo de REM.C) y de palabras gramaticales, en B y A.
FORMAS_SUFIJOS = {f for m in MORFEMAS.values()
                  for f in [m["b"], m["a"]] + ([m["alomorfo"]["b"], m["alomorfo"]["a"]] if m.get("alomorfo") else [])}
PROHIBIDAS_IGUALES = FORMAS_SUFIJOS | {p[k] for p in GRAMATICALES.values() for k in ("b", "a")}
# CH-308: sufijos de 3 grafemas o más. El "¨" no es grafema: "¨ta" tiene 2.
TERMINACIONES_PROHIBIDAS = {f for f in FORMAS_SUFIJOS
                            if len(segmentar(f.replace("¨", ""))) >= GR["terminacion_prohibida_min_grafemas"]}
# CH-310: sufijos de 1 grafema o más. Se dejan fuera los que llevan "¨": una raíz no tiene vocal larga (CH-209),
# así que r1 + "¨…" nunca puede ser otra raíz.
SUFIJOS_CH310 = {f for f in FORMAS_SUFIJOS if "¨" not in f}

if FILTRO_REAL_ACTIVO:
    print(f"{RUTA_REALES.name}: {len(REALES):,} palabras distintas · modo {FR['modo']!r} "
          f"(se descarta si n(r) ≥ {UMBRAL_REAL[FR['modo']]}).")
else:
    print(f"AVISO: falta {RUTA_REALES.name}. Las raíces se generan SIN el filtro CH-306 (pendiente, ver P-04).")
print("CH-308, terminaciones prohibidas:", " ".join(sorted(TERMINACIONES_PROHIBIDAS)))

palabras_reales.txt: 43,437 palabras distintas · modo 'guia' (se descarta si n(r) ≥ 1).
CH-308, terminaciones prohibidas: chiwi ista istu itu jmi kama kilu kiqu layku mpi mtu naka nchi nku puli pxa ska sma sulu taki tama tana thäqi tiwqa ulqa ulqu uqu ñäni


In [17]:
# 6.2 · Filtros de una raíz y generador (§3.1: CH-301 a CH-310).
import random

OCLUSIVAS_SIMPLES = {serie[0]: serie for serie in PUNTO_DE_ARTICULACION.values()}   # p → [p, ph, p']
assert all(len(s) == 3 for s in OCLUSIVAS_SIMPLES.values()), "alfabeto.yaml: cada serie es [simple, aspirada, glotalizada]"


def motivos_rechazo(r, aceptadas=()):
    """Revisa una raíz candidata con todos los filtros.

    Parámetros:
        r: raíz candidata.
        aceptadas: raíces ya aceptadas (conjunto o diccionario).
    Devuelve:
        Lista de motivos, cada uno con su regla. Vacía si la raíz se acepta.
    Reglas: CH-201 a CH-209 (validar_raiz), CH-306, CH-307, CH-308, CH-309, CH-310.
    Ejemplo:
        >>> motivos_rechazo("tisqa")
        []
        >>> motivos_rechazo("kirpuru", {"kirpu"})
        ['CH-310: kirpuru = kirpu + -ru']
    """
    motivos = validar_raiz(r)
    if FILTRO_REAL_ACTIVO and (m := motivo_real(r)):
        motivos.append(m)
    if r in PROHIBIDAS_IGUALES:
        motivos.append("CH-307: igual a un sufijo o a una palabra gramatical")
    motivos += [f"CH-308: termina en -{s}" for s in sorted(TERMINACIONES_PROHIBIDAS) if r.endswith(s)]
    if r in aceptadas:
        motivos.append("CH-309: otra entrada ya tiene esta raíz")
    for s in sorted(SUFIJOS_CH310):
        if r.endswith(s) and r[:-len(s)] in aceptadas:
            motivos.append(f"CH-310: {r} = {r[:-len(s)]} + -{s}")
        if r + s in aceptadas:
            motivos.append(f"CH-310: {r + s} = {r} + -{s}")
    return motivos


def sortear(pesos, rng):
    """Elige una clave de un diccionario {opción: peso}. Los pesos no tienen que sumar 1.

    Parámetros:
        pesos: diccionario de pesos.
        rng: generador random.Random.
    Devuelve:
        La opción elegida.
    Reglas: CH-301, CH-302, CH-303 (renormalizar = no hace falta que sumen 1).
    Ejemplo:
        >>> sortear({"a": 1.0}, random.Random(0))
        'a'
    """
    return rng.choices(list(pesos), weights=list(pesos.values()))[0]


def laringalizar(silabas, rng):
    """Paso de laringalización: a veces una oclusiva simple se vuelve aspirada o glotalizada.

    Parámetros:
        silabas: lista de sílabas [inicio, vocal, coda]; inicio y coda pueden ser "".
        rng: generador random.Random.
    Devuelve:
        Las sílabas (cambiadas o no), o None si la raíz se descarta (regla 4).
    Reglas: CH-305 (1: probabilidad 0,30; 2: aspirada o glotalizada; 3: armonía; 4: descarte), CH-205.
    Ejemplo:
        lamcha → se elige ch, glotalizada → lamch'a.
    """
    L = GR["laringalizacion"]
    if rng.random() >= L["probabilidad"]:
        return silabas
    candidatas = [s[0] for s in silabas if s[0] in OCLUSIVAS_SIMPLES]
    if not candidatas:
        return silabas
    simple = rng.choice(candidatas)
    serie = OCLUSIVAS_SIMPLES[simple]
    nueva = serie[1] if sortear(L["tipo"], rng) == "aspirada" else serie[2]
    if any(s[2] in serie for s in silabas):                      # 4: misma serie en coda → descarte
        return None
    return [[nueva if s[0] == simple else s[0], s[1], s[2]] for s in silabas]   # 3: armonía en los inicios


def candidata(cat, rng):
    """Fabrica una raíz candidata, sin filtrar.

    Parámetros:
        cat: categoría del léxico (CH-401).
        rng: generador random.Random.
    Devuelve:
        La raíz (cadena), o None si la laringalización la descarta.
    Reglas: CH-301 (sílabas), CH-302 (plantillas), CH-303 (pesos; sin x ni r en la 1.ª sílaba),
        CH-304 (verbos en -a), CH-305 (laringalización).
    Ejemplo:
        >>> candidata("V_MOV", random.Random(1))   # una raíz de 2 o 3 sílabas que termina en a
    """
    n = int(sortear(GR["silabas"][cat], rng))
    inicio_1 = {c: p for c, p in GR["inicio"].items() if c not in GR["inicio_prohibido_primera_silaba"]}
    silabas = []
    for i in range(n):
        pos = "primera" if i == 0 else "ultima" if i == n - 1 else "intermedia"
        plantilla = sortear(GR["plantillas"][pos], rng)
        inicio = sortear(inicio_1 if i == 0 else GR["inicio"], rng) if plantilla.startswith("C") else ""
        vocal = sortear(GR["vocales"], rng)
        coda = sortear(GR["coda"], rng) if plantilla.endswith("VC") else ""
        silabas.append([inicio, vocal, coda])
    if cat in VERBOS:
        silabas[-1][1] = GR["vocal_final_verbos"]               # CH-304
    silabas = laringalizar(silabas, rng)
    return None if silabas is None else "".join("".join(s) for s in silabas)


def generar_raices(lexico, semilla=GR["semilla_raices"]):
    """Da una raíz a cada entrada del léxico, recorriéndolo por id.

    Parámetros:
        lexico: DataFrame del léxico (columnas id y cat).
        semilla: semilla fija (CH-309).
    Devuelve:
        (raíces, estadística). raíces = {id: raíz}. estadística = DataFrame con los candidatos
        probados y descartados por categoría, y la regla que más descarta.
    Lanza:
        RuntimeError si una entrada no consigue raíz en max_intentos (CH-306.5).
    Reglas: §3.1 completo; CH-306.5 (aviso si se descarta más del 60 %), CH-309 (semilla, orden por id).
    Ejemplo:
        >>> generar_raices(LEXICO_EJEMPLO.head(2))[0]
        {'L0001': '…', 'L0002': '…'}
    """
    rng = random.Random(semilla)
    aceptadas, cuenta = {}, {}
    for e in lexico.sort_values("id").itertuples():
        c = cuenta.setdefault(e.cat, {"probadas": 0, "descartadas": 0, "motivos": {}})
        for _ in range(FR["max_intentos"]):
            r = candidata(e.cat, rng)
            c["probadas"] += 1
            motivos = ["CH-305: oclusiva del mismo punto en coda"] if r is None else motivos_rechazo(r, aceptadas)
            if not motivos:
                aceptadas[r] = e.id
                break
            c["descartadas"] += 1
            regla = motivos[0].split(":")[0]
            c["motivos"][regla] = c["motivos"].get(regla, 0) + 1
        else:
            raise RuntimeError(f"CH-306.5: {e.id} ({e.cat}) sin raíz tras {FR['max_intentos']} intentos.")
    estad = pd.DataFrame([{"categoría": cat, "probadas": c["probadas"], "descartadas": c["descartadas"],
                           "% descartadas": round(100 * c["descartadas"] / c["probadas"], 1),
                           "regla que más descarta": max(c["motivos"], key=c["motivos"].get) if c["motivos"] else "—"}
                          for cat, c in cuenta.items()])
    return {i: r for r, i in aceptadas.items()}, estad


print("Ejemplos de candidatas (sin filtrar):",
      ", ".join(str(candidata(c, random.Random(k))) for k, c in enumerate(CATEGORIAS)))
for r, acept in [("tisqa", {}), ("sara", {}), ("muqi", {}), ("kurnaka", {}), ("kirpuru", {"kirpu"})]:
    print(f"{r:8} → {motivos_rechazo(r, acept) or 'aceptada'}")

Ejemplos de candidatas (sin filtrar): aqsuji, lajnu, palnamma, qipa, qasa, luya, sapjula, maqa
tisqa    → aceptada
sara     → ['CH-306: es una palabra real']
muqi     → ['CH-306: 2 palabras reales empiezan por muqi (modo guia, umbral 1)', 'CH-307: igual a un sufijo o a una palabra gramatical']
kurnaka  → ['CH-308: termina en -naka']
kirpuru  → ['CH-310: kirpuru = kirpu + -ru']


In [18]:
# 6.3 · Raíces del léxico y escritura de lexico.csv.
if LEXICO_PRODUCCION:
    RAICES, ESTADISTICA = generar_raices(LEXICO)
    LEXICO["raiz"] = LEXICO["id"].map(RAICES)
    print(f"Raíces generadas con la semilla {GR['semilla_raices']} (CH-309).")
    display(ESTADISTICA)
    for f in ESTADISTICA.itertuples():
        if f[4] > 100 * FR["aviso_si_rechazo_mayor_que"]:
            print(f"AVISO CH-306.5: en {f.categoría} se descartó el {f[4]} % de los candidatos.")
else:
    # P-05 (a): las raíces de §4.1 se conservan; se revisan en orden de id, como si se aceptaran una a una.
    aceptadas, fallas = set(), {}
    for f in LEXICO.sort_values("id").itertuples():
        if m := motivos_rechazo(f.raiz, aceptadas):
            fallas[f.id] = m
        aceptadas.add(f.raiz)
    if fallas:
        raise ValueError(f"Raíces de §4.1 que no pasan los filtros: {fallas}")
    print(f"Las {len(LEXICO)} raíces de §4.1 pasan todos los filtros (CH-201 a CH-310"
          f"{'' if FILTRO_REAL_ACTIVO else ', salvo CH-306: falta la lista'}).")

LEXICO["silabas"] = LEXICO["raiz"].map(lambda r: str(silabas(r)))
assert LEXICO["raiz"].is_unique, "CH-309: dos sentidos comparten raíz"
RUTA_LEXICO = CARPETA_SALIDA / "lexico.csv"
LEXICO.to_csv(RUTA_LEXICO, index=False, encoding="utf-8")
print(f"Escrito {RUTA_LEXICO} ({len(LEXICO)} filas).")
display(LEXICO[["id", "cat", "es_m", "raiz", "silabas"]].head(25))

Las 25 raíces de §4.1 pasan todos los filtros (CH-201 a CH-310).
Escrito /Users/mazenabuhamdan/Desktop/Chulupi-lang/salida/lexico.csv (25 filas).


,id,cat,es_m,raiz,silabas
0,L0001,N_HUM,niño,lisqu,2
1,L0002,N_HUM,hermano,pultiya,3
2,L0003,N_HUM,maestro,yaruqhi,3
3,L0004,N_HUM,mujer,simlla,2
4,L0005,N_NOHUM,casa,chimula,3
5,L0006,N_NOHUM,pan,q'aspi,2
6,L0007,N_NOHUM,pueblo,wirkha,2
7,L0008,N_NOHUM,perro,kirpu,2
8,L0009,N_NOHUM,mercado,pinqara,3
9,L0010,N_NOHUM,palo,salmi,2


### Pruebas de la sección 6

- **CH-306 a CH-310:** los ejemplos de la especificación (`sara`, `kulla`, `lapi`, `tisqa`, `muqi`, `naya`, `kurnaka`, `pilampi`, `wisulu`, `chimuka`, `kirpuru`, `kirpullu`).
- **CH-301 a CH-305:** se fabrican miles de candidatas y se mira que tengan 2 o 3 sílabas, que los verbos acaben en `a` y que la laringalización cumpla CH-205.
- **CH-309:** con la misma semilla salen las mismas raíces, y nunca dos sentidos con la misma.

In [19]:
# 6.4 · Pruebas de las raíces.
class PruebasRaices(unittest.TestCase):
    @unittest.skipUnless(FILTRO_REAL_ACTIVO, "falta palabras_reales.txt")
    def test_CH306_ejemplos(self):
        """CH-306: sara ✗ (real); kulla ✗ y lapi ✗ en modo guía (inicio de kullaka, lapisa); tisqa ✓."""
        self.assertIn("palabra real", motivo_real("sara", "guia"))
        self.assertIn("palabra real", motivo_real("sara", "corpus"))
        self.assertIsNotNone(motivo_real("kulla", "guia"))
        self.assertIsNotNone(motivo_real("lapi", "guia"))
        self.assertIsNone(motivo_real("tisqa", "guia"))
        self.assertIsNone(motivo_real("tisqa", "corpus"))

    @unittest.skipUnless(FILTRO_REAL_ACTIVO, "falta palabras_reales.txt")
    def test_CH306_umbral_corpus(self):
        """CH-306.3: en modo corpus, una raíz con menos de 5 palabras que empiezan por ella pasa."""
        n = n_prefijo("lapi")
        self.assertEqual(motivo_real("lapi", "corpus") is None, n < 5)

    def test_CH306_normalizacion(self):
        """CH-306.2: NFC, minúsculas, apóstrofo U+0027 y corte por lo que no es letra."""
        self.assertEqual(tokens_reales("Sara-ña, ch’uxña (t'ant'a)"), {"sara", "ña", "ch'uxña", "t'ant'a"})

    def test_CH307(self):
        """CH-307: muqi (pronombre 1, modo A) y naya (pronombre 1, modo B) ✗."""
        self.assertTrue(tiene_regla(motivos_rechazo("muqi"), "CH-307"))
        self.assertTrue(tiene_regla(motivos_rechazo("naya"), "CH-307"))

    def test_CH308_lista(self):
        """CH-308: la lista calculada desde morfemas.yaml es la de la especificación."""
        b = "naka taki kama layku tana tama ista istu itu ñäni pxa ska sma mpi".split()
        a = "sulu puli chiwi tiwqa kilu kiqu ulqa ulqu uqu thäqi mtu nchi jmi nku".split()
        self.assertEqual(TERMINACIONES_PROHIBIDAS, set(b + a))

    def test_CH308_ejemplos(self):
        """CH-308: kurnaka, pilampi, wisulu ✗; chimuka y chimula ✓."""
        for r in ["kurnaka", "pilampi", "wisulu"]:
            self.assertTrue(tiene_regla(motivos_rechazo(r), "CH-308"), r)
        for r in ["chimuka", "chimula"]:
            self.assertFalse(tiene_regla(motivos_rechazo(r), "CH-308"), r)

    def test_CH309_repetida(self):
        """CH-309: una raíz ya aceptada no se acepta otra vez."""
        self.assertTrue(tiene_regla(motivos_rechazo("tisqa", {"tisqa"}), "CH-309"))

    def test_CH310(self):
        """CH-310: con kirpu, se rechazan kirpuru (B -ru) y kirpullu (A -llu); y al revés."""
        self.assertTrue(tiene_regla(motivos_rechazo("kirpuru", {"kirpu"}), "CH-310"))
        self.assertTrue(tiene_regla(motivos_rechazo("kirpullu", {"kirpu"}), "CH-310"))
        self.assertTrue(tiene_regla(motivos_rechazo("kirpu", {"kirpuru"}), "CH-310"))
        self.assertFalse(tiene_regla(motivos_rechazo("kirpu", {"tisqa"}), "CH-310"))

    def test_CH301_a_CH305_candidatas(self):
        """CH-301 a CH-305: 4.000 candidatas por categoría (sin filtrar)."""
        for cat in CATEGORIAS:
            rng = random.Random(7)
            cands = [c for c in (candidata(cat, rng) for _ in range(4000)) if c is not None]
            dos = sum(silabas(c) == 2 for c in cands) / len(cands)
            with self.subTest(cat=cat):
                self.assertTrue(all(silabas(c) in (2, 3) for c in cands))                    # CH-301
                self.assertAlmostEqual(dos, GR["silabas"][cat][2], delta=0.04)                # CH-301: proporción
                self.assertTrue(all(RE_RAIZ.match(clases(segmentar(c))) for c in cands))      # CH-201, CH-302
                self.assertFalse(any(segmentar(c)[0] in ("x", "r") for c in cands))           # CH-303
                self.assertFalse(any(tiene_regla(validar_raiz(c), "CH-205") for c in cands))  # CH-305
                if cat in VERBOS:
                    self.assertTrue(all(c.endswith("a") for c in cands))                      # CH-304
                lar = sum(any(g in LARINGALIZADOS for g in segmentar(c)) for c in cands) / len(cands)
                self.assertTrue(0.10 < lar < 0.30, f"laringalizadas: {lar:.2f}")              # CH-305: p = 0,30

    def test_CH309_generador(self):
        """CH-309: el generador da una raíz válida y distinta a cada entrada, y es reproducible."""
        lex = LEXICO_EJEMPLO.drop(columns="raiz")
        r1, estad = generar_raices(lex)
        self.assertEqual(r1, generar_raices(lex)[0])
        self.assertNotEqual(r1, generar_raices(lex, semilla=1)[0])
        self.assertEqual(sorted(r1), sorted(lex["id"]))
        self.assertEqual(len(set(r1.values())), len(r1))
        for i, r in r1.items():
            otras = {x for j, x in r1.items() if j != i}
            self.assertEqual(motivos_rechazo(r, otras), [], f"{i}: {r}")
        verbos = lex.loc[lex["cat"].isin(VERBOS), "id"]
        self.assertTrue(all(r1[i].endswith("a") for i in verbos))
        print(f"\n   (raíces de prueba para §4.1: {', '.join(list(r1.values())[:8])} …)", end=" ")


correr_pruebas(PruebasRaices, titulo="raíces (§3)")

test_CH301_a_CH305_candidatas (__main__.PruebasRaices.test_CH301_a_CH305_candidatas)
CH-301 a CH-305: 4.000 candidatas por categoría (sin filtrar). ... 

ok


test_CH306_ejemplos (__main__.PruebasRaices.test_CH306_ejemplos)
CH-306: sara ✗ (real); kulla ✗ y lapi ✗ en modo guía (inicio de kullaka, lapisa); tisqa ✓. ... 

ok


test_CH306_normalizacion (__main__.PruebasRaices.test_CH306_normalizacion)
CH-306.2: NFC, minúsculas, apóstrofo U+0027 y corte por lo que no es letra. ... 

ok


... (12 outputs left out)

test_CH310 (__main__.PruebasRaices.test_CH310)
CH-310: con kirpu, se rechazan kirpuru (B -ru) y kirpullu (A -llu); y al revés. ... 

ok


----------------------------------------------------------------------
Ran 10 tests in 1.208s

OK


✔ raíces (§3): 10 pruebas en verde.
